# 02 — Translation Pipeline Overview

Descriptive statistics on the full translation pipeline output for the current target term, service set, language set, and prompt variants. This notebook establishes the empirical baseline before the disagreement analysis and also stages the automated review signal files used downstream.

It asks:

- How many languages have translations from each service, and how does coverage distribute across language families?
- Which services fail on which families, and why? What do the error logs reveal about failure modes?
- Which service, language-family, and prompt cells are missing or error-prone?
- What script and directionality anomalies arise, and how are they curated?
- What automated review signals are exported for manual review and downstream analysis?


## Unit of Analysis

Different parts of this notebook use different denominators, so the first live inventory cell reports them explicitly.

- **Community translation and MT baselines** are prompt-invariant: Wikipedia interlanguage-link labels and direct MT services are counted once from the `minimal` variant.
- **LLM services** vary by prompt: coverage and quality checks can be counted per language, per service, and per prompt variant.
- **Automated review signals** collapse service/variant-level anomalies into language-level evidence for review.
- **Manual exclusions** are human review decisions applied after the automated review signals; they control what later notebooks should exclude from analysis, search-term generation, or term correction.


In [1]:
import os
import sys
import pandas as pd
import altair as alt
from pathlib import Path

alt.data_transformers.enable("vegafusion")

sys.path.insert(0, str(Path("..").resolve()))
from scripts.utils import get_data_directory_path, read_csv_file, get_language_family
from scripts.exploration.explore_confidence_within_variant import load_variant_df

DATA_DIR = get_data_directory_path()
TERMS = ["Digital Humanities"]
TARGET_TERMS = TERMS
VARIANTS = ["minimal", "fluent_speaker", "github_searcher", "judge"]

COMMUNITY_TRANSLATION_SERVICES = {
    "Wikipedia": "wikipedia_translated_term",
}
MT_BASELINE_SERVICES = {
    "Google Translate": "gt_translated_term",
    "EasyNMT": "enmt_translated_term",
    "Lingvanex": "lingvanex_translated_term",
}
PROMPT_INVARIANT_SERVICES = {**COMMUNITY_TRANSLATION_SERVICES, **MT_BASELINE_SERVICES}

# Backward-compatible alias used by older cells; notebook prose avoids calling
# Wikipedia a baseline.
BASELINE_SERVICES = PROMPT_INVARIANT_SERVICES

LLM_SERVICES = {
    "OpenAI":   "openai_translated_term",
    "Claude":   "claude_translated_term",
    "Gemini":   "gemini_translated_term",
    "DeepSeek": "deepseek_translated_term",
    "Llama":    "llama_translated_term",
    "Gemma":    "gemma_translated_term",
    "Qwen":     "qwen_translated_term",
    "Mistral":  "mistral_translated_term",
}
SERVICE_COLS = {**BASELINE_SERVICES, **LLM_SERVICES}

print(f"Data directory: {DATA_DIR}")


Retrieving translation pipeline data directory path...

Data directory: /Users/zleblanc/CodingDH/translation_transmogrification_pipeline/datasets


In [2]:
def load_all_variants(data_dir, term, variants=VARIANTS):
    """Merge per-service files for all variants into one concatenated DataFrame."""
    dfs = []
    term_slug = term.lower().replace(" ", "_")
    for variant in variants:
        df = load_variant_df(data_dir, term_slug, variant)
        if df is not None:
            df["term_source_query"] = term
            dfs.append(df)
        else:
            print(f"  No data for variant: {variant}")
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

all_dfs = {term: load_all_variants(DATA_DIR, term) for term in TARGET_TERMS}
for term, df in all_dfs.items():
    print(f"{term}: {len(df):,} rows across {df['prompt_variant'].nunique()} variants")

term = TARGET_TERMS[0]
term_slug = term.lower().replace(" ", "_")
all_terms_df = all_dfs[term]
ref = all_terms_df[all_terms_df["prompt_variant"] == VARIANTS[0]].copy()
n_lang = ref["language_code"].nunique()

community_cols = {s: c for s, c in COMMUNITY_TRANSLATION_SERVICES.items() if c in ref.columns}
mt_baseline_cols = {s: c for s, c in MT_BASELINE_SERVICES.items() if c in ref.columns}
prompt_invariant_cols = {**community_cols, **mt_baseline_cols}
llm_cols = {s: c for s, c in LLM_SERVICES.items() if c in all_terms_df.columns}

prompt_invariant_possible = n_lang * len(prompt_invariant_cols)
prompt_invariant_present = int(sum(ref[col].notna().sum() for col in prompt_invariant_cols.values()))

llm_possible = 0
llm_present = 0
llm_variant_rows = []
for variant in VARIANTS:
    vdf = all_terms_df[all_terms_df["prompt_variant"] == variant]
    v_lang = vdf["language_code"].nunique()
    v_possible = v_lang * len(llm_cols)
    v_present = int(sum(vdf[col].notna().sum() for col in llm_cols.values()))
    llm_possible += v_possible
    llm_present += v_present
    llm_variant_rows.append({
        "variant": variant,
        "languages": v_lang,
        "llm_translation_cells_filled": v_present,
        "llm_translation_cells_possible": v_possible,
        "fill_rate": v_present / v_possible if v_possible else 0,
    })

inventory_rows = [
    {"unit": "languages", "scope": "minimal variant language rows", "count": n_lang},
    {"unit": "prompt variants", "scope": "loaded variants", "count": all_terms_df["prompt_variant"].nunique()},
    {"unit": "community translation sources", "scope": "Wikipedia interlanguage-link labels", "count": len(community_cols)},
    {"unit": "MT baseline services", "scope": "prompt-invariant MT columns present", "count": len(mt_baseline_cols)},
    {"unit": "LLM services", "scope": "prompt-varying columns present", "count": len(llm_cols)},
    {"unit": "prompt-invariant translation cells", "scope": "community + MT sources, minimal variant only", "count": prompt_invariant_present, "possible": prompt_invariant_possible, "fill_rate": prompt_invariant_present / prompt_invariant_possible if prompt_invariant_possible else 0},
    {"unit": "LLM translation cells", "scope": "language x service x variant", "count": llm_present, "possible": llm_possible, "fill_rate": llm_present / llm_possible if llm_possible else 0},
]

eval_dir = os.path.join(DATA_DIR, "translated_terms", term_slug, "evaluation")
signals_path = os.path.join(eval_dir, "automated_review_signals.csv")
manual_path = os.path.join(eval_dir, "manual_exclusions.csv")
if os.path.exists(signals_path):
    _signal_rows = len(pd.read_csv(signals_path, converters={"language_code": str}))
    inventory_rows.append({"unit": "automated review signal rows", "scope": "language-level review evidence", "count": _signal_rows})
if os.path.exists(manual_path):
    _manual_rows = len(pd.read_csv(manual_path, dtype=str).fillna(""))
    inventory_rows.append({"unit": "manual review rows", "scope": "manual_exclusions.csv", "count": _manual_rows})

inventory_df = pd.DataFrame(inventory_rows)
print("Data inventory and denominators:")
display(inventory_df)

llm_variant_df = pd.DataFrame(llm_variant_rows)
print("LLM coverage denominator by prompt variant:")
display(llm_variant_df)


Digital Humanities: 3,524 rows across 4 variants
Data inventory and denominators:


,unit,scope,count,possible,fill_rate
0,languages,minimal variant language rows,881,NaN,NaN
1,prompt variants,loaded variants,4,NaN,NaN
2,community translation sources,Wikipedia interlanguage-link labels,1,NaN,NaN
3,MT baseline services,prompt-invariant MT columns present,3,NaN,NaN
4,LLM services,prompt-varying columns present,8,NaN,NaN
5,prompt-invariant translation cells,"community + MT sources, minimal variant only",491,3524.0,0.139330
6,LLM translation cells,language x service x variant,27451,28192.0,0.973716
7,automated review signal rows,language-level review evidence,881,NaN,NaN
8,manual review rows,manual_exclusions.csv,599,NaN,NaN


LLM coverage denominator by prompt variant:


,variant,languages,llm_translation_cells_filled,llm_translation_cells_possible,fill_rate
0,minimal,881,6750,7048,0.957719
1,fluent_speaker,881,6775,7048,0.961266
2,github_searcher,881,6893,7048,0.978008
3,judge,881,7033,7048,0.997872


## 2.1 Pipeline Coverage

### Service Availability Snapshot

How much of the pipeline has at least one translation from each service? Wikipedia is treated as a community translation source, while Google Translate, EasyNMT, and Lingvanex are MT baselines. All four are prompt-invariant and counted once from the `minimal` variant. LLM services are collapsed across prompt variants here: a language counts as covered if the service produced a translation in any prompt variant. Prompt-specific missingness is handled later in the service × prompt outcome matrix.


In [3]:
term = TARGET_TERMS[0]
df = all_dfs[term].copy()
ref = df[df["prompt_variant"] == VARIANTS[0]].copy()
total = ref["language_code"].nunique()

coverage_rows = []

# Community translation and MT baselines are prompt-invariant, so count
# them once from the minimal-variant reference frame.
for service, col in PROMPT_INVARIANT_SERVICES.items():
    n = int(ref[col].notna().sum()) if col in ref.columns else 0
    coverage_rows.append({
        "service": service,
        "service_class": "Community translation" if service in COMMUNITY_TRANSLATION_SERVICES else "MT baseline",
        "languages_covered": n,
        "total_languages": total,
        "coverage_rate": n / total if total else 0,
    })

# LLM services vary by prompt. For this overview snapshot, collapse to any
# successful prompt variant; prompt-specific counts appear in the service × prompt outcome matrix below.
for service, col in LLM_SERVICES.items():
    if col in df.columns:
        any_by_lang = df.groupby("language_code")[col].agg(lambda x: x.notna().any())
        n = int(any_by_lang.sum())
    else:
        n = 0
    coverage_rows.append({
        "service": service,
        "service_class": "LLM: any prompt",
        "languages_covered": n,
        "total_languages": total,
        "coverage_rate": n / total if total else 0,
    })

service_coverage = pd.DataFrame(coverage_rows)
service_order = (
    service_coverage.sort_values(["service_class", "languages_covered"], ascending=[True, False])
    ["service"].tolist()
)

coverage_chart = alt.Chart(service_coverage).mark_bar().encode(
    y=alt.Y("service:N", sort=service_order, title=None),
    x=alt.X("languages_covered:Q", scale=alt.Scale(domain=[0, total]), title="languages covered"),
    color=alt.Color(
        "service_class:N",
        scale=alt.Scale(domain=["Community translation", "MT baseline", "LLM: any prompt"], range=["#388e3c", "#8d6e63", "#1976d2"]),
        title=None,
    ),
    tooltip=[
        "service:N",
        "service_class:N",
        alt.Tooltip("languages_covered:Q", title="languages covered"),
        alt.Tooltip("total_languages:Q", title="total languages"),
        alt.Tooltip("coverage_rate:Q", format=".1%", title="coverage rate"),
    ],
).properties(width=430, height=300, title=f"Service availability snapshot ({total:,} languages)")

coverage_text = coverage_chart.mark_text(align="left", dx=4, fontSize=10).encode(
    text="languages_covered:Q",
    color=alt.value("black"),
)

display(coverage_chart + coverage_text)
display(
    service_coverage.sort_values("languages_covered", ascending=False)
    .style.format({"coverage_rate": "{:.1%}"})
)


alt.LayerChart(...)

,service,service_class,languages_covered,total_languages,coverage_rate
4,OpenAI,LLM: any prompt,881,881,100.0%
5,Claude,LLM: any prompt,881,881,100.0%
6,Gemini,LLM: any prompt,881,881,100.0%
7,DeepSeek,LLM: any prompt,881,881,100.0%
8,Llama,LLM: any prompt,881,881,100.0%
9,Gemma,LLM: any prompt,881,881,100.0%
10,Qwen,LLM: any prompt,881,881,100.0%
11,Mistral,LLM: any prompt,881,881,100.0%
1,Google Translate,MT baseline,242,881,27.5%
3,Lingvanex,MT baseline,109,881,12.4%


### Coverage by Language Family

How does each service's coverage distribute across language families? A cell is green if the service translated at least one prompt variant for that family; white/empty means no coverage. LLM services aggregate across all 4 prompt variants (any variant = covered); the MT baselines and Wikipedia community-translation source each run once per language.

In [4]:
HEATMAP_SERVICES = {
    "Wikipedia":        "wikipedia_translated_term",
    "Google Translate": "gt_translated_term",
    "EasyNMT":          "enmt_translated_term",
    "Lingvanex":        "lingvanex_translated_term",
    "OpenAI":           "openai_translated_term",
    "Claude":           "claude_translated_term",
    "Gemini":           "gemini_translated_term",
    "DeepSeek":         "deepseek_translated_term",
    "Llama":            "llama_translated_term",
    "Gemma":            "gemma_translated_term",
    "Qwen":             "qwen_translated_term",
    "Mistral":          "mistral_translated_term",
}
service_order = list(HEATMAP_SERVICES.keys())

term = TARGET_TERMS[0]
all_variants = all_dfs[term].copy()
all_variants["language_family"] = all_variants["language_code"].apply(get_language_family)

# Max coverage across all prompt variants: a language counts as covered by a
# service if any variant produced a valid translation for it.
avail_cols = {svc: col for svc, col in HEATMAP_SERVICES.items() if col in all_variants.columns}
coverage_by_lang = (
    all_variants
    .assign(**{col: all_variants[col].notna() for col in avail_cols.values()})
    .groupby("language_code")
    .agg({col: "max" for col in avail_cols.values()} | {"language_family": "first"})
    .reset_index()
)

other_langs = coverage_by_lang[coverage_by_lang["language_family"] == "Other"]["language_code"].tolist()
if other_langs:
    print(f"⚠ Ungrouped languages: {other_langs}")
else:
    print("✓ All languages assigned to a family")

family_counts = coverage_by_lang["language_family"].value_counts().to_dict()

rows = []
for family, n_fam in sorted(family_counts.items(), key=lambda x: -x[1]):
    fam_df = coverage_by_lang[coverage_by_lang["language_family"] == family]
    label = f"{family} ({n_fam})"
    for service, col in HEATMAP_SERVICES.items():
        n_translated = int(fam_df[col].sum()) if col in fam_df.columns else 0
        pct = round(n_translated / n_fam * 100, 1) if n_fam else 0.0
        rows.append({"Family": label, "Service": service, "Coverage": pct, "N": n_translated, "Family_size": n_fam})
coverage_long = pd.DataFrame(rows)

family_order = [f"{f} ({family_counts[f]})" for f in sorted(family_counts.keys(), key=lambda k: -family_counts[k])]
chart = alt.Chart(coverage_long).mark_rect().encode(
    x=alt.X("Service:N", sort=service_order, title="Translation Service"),
    y=alt.Y("Family:N", sort=family_order, title="Language Family"),
    color=alt.Color("Coverage:Q",
        scale=alt.Scale(scheme="yellowgreen", domain=[0, 100]),
        legend=alt.Legend(title="% coverage"),
    ),
    tooltip=[
        "Family", "Service",
        alt.Tooltip("Coverage:Q", format=".1f", title="% coverage"),
        alt.Tooltip("N:Q", title="languages translated"),
        alt.Tooltip("Family_size:Q", title="family size"),
    ],
).properties(
    title=alt.Title(
        "Languages Translated by Service and Language Family",
        subtitle="LLM services: any prompt variant = covered. Wikipedia community translation and MT baselines run once.",
    ),
    width=520, height=720,
)
text = chart.mark_text(baseline="middle", fontSize=8).encode(
    text=alt.Text("Coverage:Q", format=".0f"),
    color=alt.condition(alt.datum.Coverage > 60, alt.value("white"), alt.value("black")),
)
(chart + text)

Retrieving translation pipeline data directory path...

✓ All languages assigned to a family


alt.LayerChart(...)

### Coverage Depth by Language

Which languages are still weakly covered after collapsing across services and prompt variants? This section keeps the focus on pipeline availability: zero- and sparse-coverage languages are candidates for closer review in the error, quality, and downstream search notebooks.


In [5]:
term = TARGET_TERMS[0]
df_all = all_dfs[term].copy()
df_all["language_family"] = df_all["language_code"].apply(get_language_family)

all_svc_cols = {**PROMPT_INVARIANT_SERVICES, **LLM_SERVICES}
svc_cols = [c for c in all_svc_cols.values() if c in df_all.columns]

# A service "covers" a language if it succeeded in at least one prompt variant
coverage = df_all.groupby("language_code")[svc_cols].agg(lambda x: x.notna().any()).reset_index()
lang_meta = df_all[["language_code", "language_name", "language_family"]].drop_duplicates("language_code")
coverage_depth = coverage.merge(lang_meta, on="language_code")
coverage_depth["n_services"] = coverage_depth[svc_cols].sum(axis=1)

TIERS = {"0 — none": (0,0), "1–2 — sparse": (1,2), "3–5 — partial": (3,5), "6–9 — rich": (6,9)}
def tier(n):
    for label, (lo, hi) in TIERS.items():
        if lo <= n <= hi: return label
    return "other"
coverage_depth["tier"] = coverage_depth["n_services"].apply(tier)

tier_order = list(TIERS.keys())
hist = alt.Chart(coverage_depth).mark_bar().encode(
    x=alt.X("n_services:O", title="Services that translated this language (any prompt)", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("count():Q", title="Number of languages"),
    color=alt.Color("tier:N", sort=tier_order,
        scale=alt.Scale(domain=tier_order, range=["#d32f2f","#ff9800","#1976d2","#388e3c"]),
        title="Coverage tier"),
    tooltip=["n_services:O", "count():Q", "tier:N"],
).properties(width=360, height=240, title="Service coverage depth per language (best across all prompts)")

low_cov = coverage_depth[coverage_depth["n_services"] <= 3][
    ["language_code","language_name","language_family","n_services","tier"]
].sort_values(["n_services","language_family"]).reset_index(drop=True)

print(f"Zero coverage: {(coverage_depth['n_services']==0).sum()} | Sparse (1-3): {((coverage_depth['n_services']>=1)&(coverage_depth['n_services']<=3)).sum()}. Only sparse languages: {low_cov['language_name'].tolist()}")

hist

Zero coverage: 0 | Sparse (1-3): 0. Only sparse languages: []


alt.Chart(...)

### Error Log Analysis

What do the current error logs tell us about *why* translations failed?

This section uses only the current prompt variants: `minimal`, `fluent_speaker`, `github_searcher`, and `judge`. Older prompt-variant rows have been removed from the LLM error-log CSVs so the counts here match the current experiment design.

The first chart preserves raw logged error counts as a pipeline artifact. The LLM analysis then uses two levels of description:

- **`failure_layer`** — where the failure happened in the pipeline: request/provider failure, parse failure, empty output, model refusal, model uncertainty, generation artifact, or unclassified.
- **`category`** — the more specific diagnostic label: `api_error`, `max_tokens`, `parse_format`, `knowledge_gap`, `repetition_loop`, and so on.

The final audit table normalizes repeated messages into `message_signature`s. This keeps raw counts visible while showing whether a category is driven by one repeated template or by many distinct failure patterns.


In [6]:
SERVICE_NAMES = {
    "claude":    "Claude",   "deepseek": "DeepSeek", "enmt":  "EasyNMT",
    "gemini":    "Gemini",   "gemma":    "Gemma",    "gt":    "Google Translate",
    "lingvanex": "Lingvanex","llama":    "Llama",    "mistral": "Mistral",
    "openai":    "OpenAI",   "qwen":     "Qwen",     "wikipedia": "Wikipedia",
}

error_dir = os.path.join(DATA_DIR, "error_logs")
error_rows = []

if os.path.exists(error_dir):
    for fname in sorted(os.listdir(error_dir)):
        if not fname.endswith(".csv"): continue
        key = fname.replace("translation_errors.csv", "")
        name = SERVICE_NAMES.get(key, key)
        try:
            edf = read_csv_file(os.path.join(error_dir, fname))
            if "status_code" in edf.columns:
                for code, cnt in edf["status_code"].value_counts().items():
                    error_rows.append({"service": name, "status": str(code), "count": int(cnt)})
            else:
                error_rows.append({"service": name, "status": "unknown", "count": len(edf)})
        except Exception as e:
            print(f"Could not read {fname}: {e}")

error_df = pd.DataFrame(error_rows)

if error_df.empty:
    print("No error rows found — skipping error charts.")
else:
    totals = error_df.groupby("service")["count"].sum().reset_index().sort_values("count", ascending=False)
    svc_order = totals["service"].tolist()

    total_bar = alt.Chart(totals).mark_bar().encode(
        y=alt.Y("service:N", sort=svc_order, title=None),
        x=alt.X("count:Q", title="total errors logged"),
        color=alt.Color("service:N", legend=None, scale=alt.Scale(scheme="tableau10")),
        tooltip=["service:N", "count:Q"],
    ).properties(width=320, height=220, title="Total errors per service")

    total_text = total_bar.mark_text(align="left", dx=4, fontSize=9).encode(
        text="count:Q", color=alt.value("black"))

    status_bar = alt.Chart(error_df).mark_bar().encode(
        y=alt.Y("service:N", sort=svc_order, title=None),
        x=alt.X("count:Q", title="error count"),
        color=alt.Color("status:N", title="HTTP status", scale=alt.Scale(scheme="set2")),
        order=alt.Order("count:Q", sort="descending"),
        tooltip=["service:N", "status:N", "count:Q"],
    ).properties(width=320, height=220, title="Error breakdown by status code")

    display((total_bar + total_text) | status_bar)

alt.HConcatChart(...)

In [7]:
PARSE_PREFIX = "Could not parse translation response: "
LLM_ERROR_FILES = {
    "Claude":   "claude_translation_errors.csv",
    "OpenAI":   "openai_translation_errors.csv",
    "Gemini":   "gemini_translation_errors.csv",
    "DeepSeek": "deepseek_translation_errors.csv",
    "Llama":    "llama_translation_errors.csv",
    "Gemma":    "gemma_translation_errors.csv",
    "Qwen":     "qwen_translation_errors.csv",
    "Mistral":  "mistral_translation_errors.csv",
}

CATEGORIES = [
    ("max_tokens",        ["MAX_TOKENS", "token limit", "hit token limit", "truncated"]),
    ("api_error",         ["INTERNAL", "An internal error has occurred",
                           "read operation timed out", "timeout", "timed out"]),
    ("empty_translation", ["Model returned empty translated_term"]),
    ("ollama_timeout",    ["'total_duration'"]),
    ("extinct_ancient",   ["extinct", "ancient", "undeciphered", "no native speakers",
                           "historical language", "limited corpus", "Minoan", "funerary",
                           "no longer spoken"]),
    ("knowledge_gap",     ["limited resources", "limited documentation", "limited data",
                           "limited information", "limited available",
                           "not have the ability", "not able to provide",
                           "don't have the capability", "do not have the capacity",
                           "not equipped", "knowledge cutoff", "training data",
                           "not within my capabilities", "cannot provide",
                           "unable to provide", "not sufficiently documented",
                           "not thoroughly documented", "not widely documented",
                           "under-documented", "lesser-known", "lesser-documented",
                           "less commonly", "less widely", "no comprehensive",
                           "no data on", "not have data", "do not have specific",
                           "don't have specific",
                           "does not have a direct equivalent",
                           "does not directly translate",
                           "no available data", "no available translation",
                           "no known translation",
                           "couldn't find", "could not find"]),
    ("generic_refusal",   ["I'm sorry, I can't", "I'm sorry, I cannot",
                           "I'm sorry, but",
                           "I must respectfully decline",
                           "I can't assist", "I can't comply", "I can't do that",
                           "I cannot perform", "No rationale provided",
                           "cannot fulfill", "can't fulfill", "unable to fulfill",
                           "unable to translate", "not_available"]),
]

ERROR_LAYER_MAP = {
    "api_error": "request_failure",
    "max_tokens": "request_failure",
    "ollama_timeout": "request_failure",
    "parse_format": "parse_failure",
    "empty_translation": "empty_output",
    "generic_refusal": "model_refusal",
    "knowledge_gap": "model_uncertainty",
    "extinct_ancient": "model_uncertainty",
    "repetition_loop": "generation_artifact",
    "other": "unclassified",
    "unknown": "unclassified",
}

import re as _re

def _is_repetition_loop(raw_msg: str) -> bool:
    """True if the error wraps a translated_term that is a hallucination repetition loop."""
    text = raw_msg.replace(PARSE_PREFIX, "").strip()
    text = _re.sub(r"^```json\s*", "", text).strip()
    m = _re.search(r'"translated_term"\s*:\s*"(.{15,})', text)
    if not m:
        return False
    term = m.group(1).rstrip('"} \n\r\t')
    if len(term) < 20:
        return False
    if len(set(term)) / len(term) < 0.15:
        return True
    for start in range(min(5, len(term))):
        for chunk_len in range(2, min(len(term) // 4 + 1, 12)):
            chunk = term[start : start + chunk_len]
            if chunk and term.count(chunk) >= 5:
                return True
    return False

def classify_error(msg):
    if not isinstance(msg, str) or not msg.strip():
        return "unknown"
    text = msg.replace(PARSE_PREFIX, "").strip()
    # Repetition-loop check before parse_format: the JSON may be structurally valid
    # but the translated_term is a hallucination loop.
    if _is_repetition_loop(msg):
        return "repetition_loop"
    if text.startswith("{") or '"translated_term"' in text:
        return "parse_format"
    for cat, keywords in CATEGORIES:
        if any(kw.lower() in text.lower() for kw in keywords):
            return cat
    return "other"

def message_signature(msg):
    """Normalize repeated boilerplate without hiding raw error counts."""
    if not isinstance(msg, str) or not msg.strip():
        return "<empty>"
    text = msg.replace(PARSE_PREFIX, "").strip()
    text = _re.sub(r"```json|```", "", text, flags=_re.I)
    text = _re.sub(r'"translated_term"\s*:\s*"[^"]*"', '"translated_term": "<term>"', text, flags=_re.S)
    text = _re.sub(r'"translation_rationale"\s*:\s*"[^"]*"', '"translation_rationale": "<rationale>"', text, flags=_re.S)
    text = _re.sub(r"Digital Humanities\s*→\s*[^.\n]+", "Digital Humanities → <language>", text)
    text = _re.sub(r"\b[0-9]{3,}\b", "<num>", text)
    text = _re.sub(r"(.)\1{5,}", r"\1<repeat>", text)
    text = _re.sub(r"\s+", " ", text).strip()
    return text[:180]

lang_names = (
    all_dfs[TARGET_TERMS[0]][["language_code", "language_name"]]
    .drop_duplicates("language_code")
    .set_index("language_code")["language_name"]
    .to_dict()
)

llm_frames = []
for service, fname in LLM_ERROR_FILES.items():
    path = os.path.join(error_dir, fname)
    if not os.path.exists(path):
        continue
    edf = read_csv_file(path)
    if edf.empty:
        continue
    edf["service"] = service
    edf["variant"] = edf.get("variant", "").fillna("").astype(str).str.strip()
    non_current = edf[~edf["variant"].isin(VARIANTS)]
    if not non_current.empty:
        print(f"Warning: {service} has {len(non_current)} non-current error rows still present; excluding them from Notebook 02.")
    edf = edf[edf["variant"].isin(VARIANTS)].copy()
    if edf.empty:
        continue
    edf["category"] = edf["error_message"].apply(classify_error)
    edf["failure_layer"] = edf["category"].map(ERROR_LAYER_MAP).fillna("unclassified")
    edf["clean_msg"] = edf["error_message"].str.replace(PARSE_PREFIX, "", regex=False).str.strip()
    edf["message_signature"] = edf["error_message"].apply(message_signature)
    edf["language_name"] = edf["language_code"].map(lang_names).fillna(edf["language_code"])
    llm_frames.append(edf)

llm_err = pd.concat(llm_frames, ignore_index=True) if llm_frames else pd.DataFrame()

if llm_err.empty:
    print("No current-variant LLM errors found.")
else:
    print(f"Current-variant LLM errors: {len(llm_err)} rows across {llm_err['language_code'].nunique()} unique languages")
    print(f"Unique raw error messages: {llm_err['error_message'].nunique()}")
    print(f"Unique normalized message signatures: {llm_err['message_signature'].nunique()}\n")
    print("Most repeated message signatures:")
    display(
        llm_err["message_signature"].value_counts()
        .head(10)
        .reset_index()
        .rename(columns={"message_signature": "signature", "count": "n_error_rows"})
    )


Current-variant LLM errors: 614 rows across 302 unique languages
Unique raw error messages: 531
Unique normalized message signatures: 529

Most repeated message signatures:


,signature,n_error_rows
0,"<num> INTERNAL. {'error': {'code': <num>, 'mes...",24
1,Model returned empty translated_term,23
2,<empty>,16
3,'NoneType' object has no attribute 'strip',7
4,I don't have the capability to translate langu...,5
5,"I'm sorry, but I can't assist with that request.",3
6,I don't have the capability to translate langu...,3
7,I can't fulfill that request.,3
8,I don't have the capability to translate terms...,3
9,"{""translated_term"": ""གཟའ་རིགས་རིག་གནས་རིག་གནས་...",2


In [8]:
if llm_err.empty:
    print("No LLM error categories to plot.")
else:
    cat_order = [c for c, _ in CATEGORIES] + ["repetition_loop", "parse_format", "other", "unknown"]
    layer_order = [
        "request_failure", "parse_failure", "empty_output", "model_refusal",
        "model_uncertainty", "generation_artifact", "unclassified",
    ]

    error_summary = (
        llm_err.groupby(["failure_layer", "category"], as_index=False)
        .agg(
            n_error_rows=("error_message", "size"),
            n_languages=("language_code", "nunique"),
            n_services=("service", "nunique"),
            n_message_signatures=("message_signature", "nunique"),
        )
        .sort_values("n_error_rows", ascending=False)
    )
    print("Failure layer × category summary:")
    display(error_summary)

    layer_counts = (
        llm_err.groupby("failure_layer", as_index=False)
        .agg(n_error_rows=("error_message", "size"), n_languages=("language_code", "nunique"))
    )
    layer_chart = alt.Chart(layer_counts).mark_bar().encode(
        y=alt.Y("failure_layer:N", sort=layer_order, title=None),
        x=alt.X("n_error_rows:Q", title="logged error rows"),
        color=alt.Color("failure_layer:N", sort=layer_order, legend=None, scale=alt.Scale(scheme="tableau10")),
        tooltip=["failure_layer:N", "n_error_rows:Q", "n_languages:Q"],
    ).properties(width=360, height=180, title="LLM errors by failure layer")
    layer_text = layer_chart.mark_text(align="left", dx=4, fontSize=10).encode(
        text="n_error_rows:Q", color=alt.value("black")
    )

    service_order = [s for s in LLM_SERVICES if s in llm_err["service"].unique()]
    cat_grid = pd.MultiIndex.from_product(
        [service_order, cat_order], names=["service", "category"]
    ).to_frame(index=False)
    cat_counts = (
        llm_err.groupby(["service", "category"])
        .size()
        .reset_index(name="n")
    )
    cat_counts = (
        cat_grid.merge(cat_counts, on=["service", "category"], how="left")
        .fillna({"n": 0})
    )
    cat_counts["n"] = cat_counts["n"].astype(int)

    service_category_heat = alt.Chart(cat_counts).mark_rect().encode(
        x=alt.X("category:N", sort=cat_order, title="Error category", axis=alt.Axis(labelAngle=-35)),
        y=alt.Y("service:N", sort=service_order, title="Service"),
        color=alt.Color(
            "n:Q",
            scale=alt.Scale(scheme="oranges"),
            legend=alt.Legend(title="Logged error rows"),
        ),
        tooltip=["service:N", "category:N", alt.Tooltip("n:Q", title="logged error rows")],
    ).properties(width=560, height=220, title="LLM error category matrix by service")

    service_category_text = alt.Chart(cat_counts[cat_counts["n"] > 0]).mark_text(fontSize=10).encode(
        x=alt.X("category:N", sort=cat_order),
        y=alt.Y("service:N", sort=service_order),
        text="n:Q",
        color=alt.condition(alt.datum.n > cat_counts["n"].max() * 0.45, alt.value("white"), alt.value("black")),
    )

    display((layer_chart + layer_text) | (service_category_heat + service_category_text))

    # Signature audit: category-level view of repetition vs diversity.
    sig_audit = (
        llm_err.groupby(["failure_layer", "category", "message_signature"], as_index=False)
        .agg(
            n_error_rows=("error_message", "size"),
            n_languages=("language_code", "nunique"),
            services=("service", lambda x: "; ".join(sorted(x.unique()))),
        )
        .sort_values(["category", "n_error_rows"], ascending=[True, False])
    )
    focus_categories = ["other", "parse_format", "generic_refusal", "knowledge_gap", "repetition_loop"]
    print("Top normalized message signatures for audit categories:")
    display(
        sig_audit[sig_audit["category"].isin(focus_categories)]
        .groupby("category", group_keys=False)
        .head(5)
        .reset_index(drop=True)
    )

    # Drill-down: languages in model uncertainty and generation-artifact categories.
    for cat in ("knowledge_gap", "extinct_ancient", "repetition_loop"):
        sub = llm_err[llm_err["category"] == cat][
            ["service", "language_code", "language_name", "message_signature", "clean_msg"]
        ].copy()
        if sub.empty:
            continue
        print(f"\n-- {cat} ({len(sub)} errors, {sub['language_code'].nunique()} languages) --")

        multi_svc = (
            sub.groupby(["language_code", "language_name"])
            .agg(
                n_services=("service", "nunique"),
                services=("service", lambda x: sorted(x.unique())),
            )
            .reset_index()
            .query("n_services >= 2")
            .sort_values("n_services", ascending=False)
        )
        if not multi_svc.empty:
            print(f"  Flagged by >=2 services ({len(multi_svc)} languages):")
            for _, r in multi_svc.iterrows():
                print(f"    {r['language_code']} ({r['language_name']}): {r['services']}")

        print("\n  Sample signatures:")
        for svc, grp in sub.groupby("service"):
            print(f"  [{svc}]")
            for _, row in grp.drop_duplicates("message_signature").head(2).iterrows():
                print(f"    {row['language_code']} ({row['language_name']}): {row['message_signature'][:140]}")


Failure layer × category summary:


,failure_layer,category,n_error_rows,n_languages,n_services,n_message_signatures
4,model_uncertainty,knowledge_gap,179,129,4,170
2,model_refusal,generic_refusal,125,108,3,121
8,unclassified,other,104,89,4,98
1,generation_artifact,repetition_loop,61,56,4,60
7,request_failure,max_tokens,39,34,3,34
3,model_uncertainty,extinct_ancient,37,22,5,37
6,request_failure,api_error,25,18,1,2
0,empty_output,empty_translation,23,23,3,1
9,unclassified,unknown,16,15,1,1
5,parse_failure,parse_format,5,5,3,5


alt.HConcatChart(...)

Top normalized message signatures for audit categories:


,failure_layer,category,message_signature,n_error_rows,n_languages,services
0,model_refusal,generic_refusal,I can't fulfill that request.,3,3,Llama
1,model_refusal,generic_refusal,"I'm sorry, but I can't assist with that request.",3,3,OpenAI
2,model_refusal,generic_refusal,I can't fulfill that request. I can help you w...,1,1,Llama
3,model_refusal,generic_refusal,I can't fulfill that request. I can’t provide ...,1,1,Llama
4,model_refusal,generic_refusal,I can't fulfill that request. I can’t provide ...,1,1,Llama
5,model_uncertainty,knowledge_gap,I don't have the capability to translate langu...,5,5,Llama
6,model_uncertainty,knowledge_gap,I don't have the capability to translate langu...,3,3,Llama
7,model_uncertainty,knowledge_gap,I don't have the capability to translate terms...,3,3,Llama
8,model_uncertainty,knowledge_gap,I don't have the capability to translate langu...,2,2,Llama
9,model_uncertainty,knowledge_gap,Classical Mandaic does not have a direct equiv...,1,1,Llama



-- knowledge_gap (179 errors, 129 languages) --
  Flagged by >=2 services (28 languages):
    mdt (Mbere): ['Gemini', 'Llama', 'OpenAI']
    ain (Ainu): ['Llama', 'OpenAI']
    nxq (Naxi): ['Llama', 'OpenAI']
    yrk (Nenets): ['Llama', 'OpenAI']
    xmr (Meroitic): ['Llama', 'OpenAI']
    xlc (Lycian): ['Gemini', 'OpenAI']
    xcr (Carian): ['Llama', 'OpenAI']
    was (Washo): ['Llama', 'OpenAI']
    unx (Munda): ['Llama', 'OpenAI']
    tht (Tahltan): ['Llama', 'OpenAI']
    tgx (Tagish): ['Llama', 'OpenAI']
    saz (Saurashtra): ['Llama', 'Mistral']
    sad (Sandawe): ['Llama', 'OpenAI']
    oka (Okanagan): ['Llama', 'Mistral']
    mvy (Indus Kohistani): ['Llama', 'OpenAI']
    aro (Araona): ['Gemini', 'OpenAI']
    mde (Maba): ['Llama', 'OpenAI']
    kvx (Parkari Koli): ['Llama', 'Mistral']
    kut (Kutenai): ['Llama', 'OpenAI']
    hnn (Hanunoo): ['Llama', 'OpenAI']
    gld (Nanai): ['Llama', 'OpenAI']
    gjk (Kachi Koli): ['Llama', 'OpenAI']
    fud (East Futuna): ['Mistral', 'O

#### Service × Prompt × Outcome Matrix

The previous charts count errors by service and category. This aggregate adds the prompt dimension and makes the denominator explicit: each LLM service is expected to produce one translation for each language in each prompt variant. The table below separates three related quantities:

- **Parsed translations**: cells where the loaded service/variant output contains a translation after translation-rationale pairing checks.
- **Missing translation cells**: expected service × language × variant cells with no parsed translation in the loaded data.
- **Logged errors**: request-level error rows from the service error logs, grouped by prompt variant and category.

The residual chart is descriptive rather than causal. For each error category, it compares the observed count for a service × variant cell with the count expected from that service's overall error volume and that variant's overall error volume. Positive residuals identify combinations that are overrepresented for that error type; negative residuals identify combinations that are underrepresented.


In [9]:
# Service × prompt × outcome matrix.
# This quantifies the main multiple-comparison question in Notebook 02:
# which service/prompt combinations have missing output or logged failures?
term = TARGET_TERMS[0]
df = all_dfs[term].copy()

service_order = list(LLM_SERVICES.keys())
variant_order = VARIANTS

# Error logs have been pruned to the current prompt variants, so this matrix
# can use llm_err directly.
current_llm_err = llm_err.copy()

outcome_rows = []
for variant in variant_order:
    vdf = df[df["prompt_variant"] == variant]
    possible = int(vdf["language_code"].nunique())
    for service, col in LLM_SERVICES.items():
        if col in vdf.columns:
            has_translation = vdf[col].notna() & ~vdf[col].astype(str).str.strip().isin(["", "nan"])
            parsed_translations = int(has_translation.sum())
        else:
            parsed_translations = 0
        svc_var_err = current_llm_err[
            (current_llm_err["service"] == service) &
            (current_llm_err["variant"] == variant)
        ]
        error_languages = int(svc_var_err["language_code"].nunique()) if not svc_var_err.empty else 0
        logged_errors = int(len(svc_var_err))
        missing_cells = max(possible - parsed_translations, 0)
        outcome_rows.append({
            "service": service,
            "variant": variant,
            "possible_cells": possible,
            "parsed_translations": parsed_translations,
            "missing_translation_cells": missing_cells,
            "logged_error_rows": logged_errors,
            "logged_error_languages": error_languages,
            "parsed_translation_rate": parsed_translations / possible if possible else 0,
            "missing_translation_rate": missing_cells / possible if possible else 0,
            "logged_error_language_rate": error_languages / possible if possible else 0,
        })

service_variant_outcomes = pd.DataFrame(outcome_rows)

# Error-category table with both raw rows and unique-language counts.
err_grid = pd.MultiIndex.from_product(
    [service_order, variant_order, cat_order],
    names=["service", "variant", "category"],
).to_frame(index=False)

if current_llm_err.empty:
    service_variant_errors = err_grid.assign(n_error_rows=0, n_languages=0)
else:
    err_row_counts = (
        current_llm_err.groupby(["service", "variant", "category"])
        .size()
        .reset_index(name="n_error_rows")
    )
    err_lang_counts = (
        current_llm_err.groupby(["service", "variant", "category"])["language_code"]
        .nunique()
        .reset_index(name="n_languages")
    )
    service_variant_errors = (
        err_grid.merge(err_row_counts, on=["service", "variant", "category"], how="left")
        .merge(err_lang_counts, on=["service", "variant", "category"], how="left")
        .fillna({"n_error_rows": 0, "n_languages": 0})
    )
    service_variant_errors[["n_error_rows", "n_languages"]] = service_variant_errors[["n_error_rows", "n_languages"]].astype(int)

service_variant_errors = service_variant_errors.merge(
    service_variant_outcomes[["service", "variant", "possible_cells"]],
    on=["service", "variant"],
    how="left",
)
service_variant_errors["error_rows_per_attempt"] = (
    service_variant_errors["n_error_rows"] / service_variant_errors["possible_cells"]
).fillna(0)
service_variant_errors["error_languages_per_attempt"] = (
    service_variant_errors["n_languages"] / service_variant_errors["possible_cells"]
).fillna(0)

# Standardized residuals within each error category.
residual_frames = []
for category, sub in service_variant_errors.groupby("category"):
    total_n = sub["n_error_rows"].sum()
    if total_n == 0:
        tmp = sub.copy()
        tmp["expected_error_rows"] = 0.0
        tmp["std_residual"] = 0.0
        residual_frames.append(tmp)
        continue
    service_totals = sub.groupby("service")["n_error_rows"].transform("sum")
    variant_totals = sub.groupby("variant")["n_error_rows"].transform("sum")
    tmp = sub.copy()
    tmp["expected_error_rows"] = service_totals * variant_totals / total_n
    tmp["std_residual"] = tmp.apply(
        lambda r: (r["n_error_rows"] - r["expected_error_rows"]) / (r["expected_error_rows"] ** 0.5)
        if r["expected_error_rows"] > 0 else 0,
        axis=1,
    )
    residual_frames.append(tmp)

service_variant_error_residuals = pd.concat(residual_frames, ignore_index=True)

# Save reusable summaries for later notebooks.
eval_dir = os.path.join(DATA_DIR, "translated_terms", term.lower().replace(" ", "_"), "evaluation")
os.makedirs(eval_dir, exist_ok=True)
outcome_path = os.path.join(eval_dir, "service_variant_outcomes.csv")
error_path = os.path.join(eval_dir, "service_variant_error_summary.csv")
resid_path = os.path.join(eval_dir, "service_variant_error_residuals.csv")
service_variant_outcomes.to_csv(outcome_path, index=False)
service_variant_errors.to_csv(error_path, index=False)
service_variant_error_residuals.to_csv(resid_path, index=False)

print(f"Saved: {outcome_path}")
print(f"Saved: {error_path}")
print(f"Saved: {resid_path}")
print()
print("Service × prompt outcome summary:")
display(
    service_variant_outcomes
    .sort_values(["missing_translation_rate", "logged_error_language_rate"], ascending=False)
    .style.format({
        "parsed_translation_rate": "{:.1%}",
        "missing_translation_rate": "{:.1%}",
        "logged_error_language_rate": "{:.1%}",
    })
)

# Heatmap 1: missing output by service and prompt variant.
missing_heat = alt.Chart(service_variant_outcomes).mark_rect().encode(
    x=alt.X("variant:N", sort=variant_order, title="Prompt variant"),
    y=alt.Y("service:N", sort=service_order, title="Service"),
    color=alt.Color(
        "missing_translation_rate:Q",
        scale=alt.Scale(scheme="oranges"),
        title="Missing translation rate",
        legend=alt.Legend(format="%"),
    ),
    tooltip=[
        "service:N", "variant:N",
        alt.Tooltip("possible_cells:Q", title="expected cells"),
        alt.Tooltip("parsed_translations:Q", title="parsed translations"),
        alt.Tooltip("missing_translation_cells:Q", title="missing cells"),
        alt.Tooltip("logged_error_languages:Q", title="languages with logged errors"),
        alt.Tooltip("missing_translation_rate:Q", format=".1%", title="missing rate"),
    ],
).properties(width=360, height=230, title="Missing translation cells by service × prompt")

missing_text = alt.Chart(service_variant_outcomes).mark_text(fontSize=10).encode(
    x=alt.X("variant:N", sort=variant_order),
    y=alt.Y("service:N", sort=service_order),
    text=alt.Text("missing_translation_rate:Q", format=".0%"),
    color=alt.condition(alt.datum.missing_translation_rate > 0.25, alt.value("white"), alt.value("black")),
)

display(missing_heat + missing_text)

# Heatmap 2: standardized residuals for the most common logged error categories.
top_error_categories = (
    service_variant_errors.groupby("category")["n_error_rows"].sum()
    .sort_values(ascending=False)
    .loc[lambda s: s > 0]
    .head(6)
    .index.tolist()
)

if top_error_categories:
    resid_plot_df = service_variant_error_residuals[
        service_variant_error_residuals["category"].isin(top_error_categories)
    ].copy()
    resid_chart = alt.Chart(resid_plot_df).mark_rect().encode(
        x=alt.X("variant:N", sort=variant_order, title="Prompt variant"),
        y=alt.Y("service:N", sort=service_order, title="Service"),
        color=alt.Color(
            "std_residual:Q",
            scale=alt.Scale(scheme="redblue", domainMid=0),
            title="Std. residual",
        ),
        tooltip=[
            "category:N", "service:N", "variant:N",
            alt.Tooltip("n_error_rows:Q", title="observed rows"),
            alt.Tooltip("expected_error_rows:Q", format=".2f", title="expected rows"),
            alt.Tooltip("std_residual:Q", format=".2f", title="std. residual"),
        ],
    ).properties(width=180, height=190).facet(
        facet=alt.Facet("category:N", sort=top_error_categories, title="Error category"),
        columns=3,
        title="Over/under-represented service × prompt combinations by error type",
    )
    display(resid_chart)
else:
    print("No current-variant LLM errors available for residual plotting.")


Saved: /Users/zleblanc/CodingDH/translation_transmogrification_pipeline/datasets/translated_terms/digital_humanities/evaluation/service_variant_outcomes.csv
Saved: /Users/zleblanc/CodingDH/translation_transmogrification_pipeline/datasets/translated_terms/digital_humanities/evaluation/service_variant_error_summary.csv
Saved: /Users/zleblanc/CodingDH/translation_transmogrification_pipeline/datasets/translated_terms/digital_humanities/evaluation/service_variant_error_residuals.csv

Service × prompt outcome summary:


,service,variant,possible_cells,parsed_translations,missing_translation_cells,logged_error_rows,logged_error_languages,parsed_translation_rate,missing_translation_rate,logged_error_language_rate
8,OpenAI,fluent_speaker,881,631,250,131,131,71.6%,28.4%,14.9%
20,Llama,github_searcher,881,786,95,87,87,89.2%,10.8%,9.9%
4,Llama,minimal,881,787,94,94,94,89.3%,10.7%,10.7%
0,OpenAI,minimal,881,797,84,90,90,90.5%,9.5%,10.2%
7,Mistral,minimal,881,829,52,52,52,94.1%,5.9%,5.9%
2,Gemini,minimal,881,846,35,40,40,96.0%,4.0%,4.5%
18,Gemini,github_searcher,881,848,33,23,23,96.3%,3.7%,2.6%
5,Gemma,minimal,881,865,16,16,16,98.2%,1.8%,1.8%
10,Gemini,fluent_speaker,881,865,16,10,10,98.2%,1.8%,1.1%
3,DeepSeek,minimal,881,868,13,17,17,98.5%,1.5%,1.9%


alt.LayerChart(...)

alt.FacetChart(...)

The service × prompt matrix above stays at the level of pipeline accounting: which cells are present, missing, or logged as failures. Notebook 04 picks up the same service × prompt grid as an interpretive object, asking how usable LLM outputs vary across prompts and services once the raw failure surface is known.


#### Repetition-Loop Language Profile

Some LLM errors show a distinctive failure pattern: the model produces syntactically valid JSON but the `translated_term` value is a short syllable or character sequence repeated until the field is hundreds of characters long. This is a model-side generation artifact, not a prompt or parsing problem, and is detected by the `_is_repetition_loop()` check applied before all other classifiers.

Two structural drivers appear:
- **Low-resource phonology**: languages with complex tone or nasal-vowel sequences where the model locks on to a recurrent phoneme.
- **Rare or extinct scripts**: cuneiform, Sogdian, Meroitic, Linear A, and related low-data scripts where the model has little target-script evidence.

Languages flagged by multiple services are the pipeline's hardest cases: translation is unreliable regardless of service.


In [10]:
# Repetition-loop language profile
rep = llm_err[llm_err["category"] == "repetition_loop"].copy()

print(f"Repetition-loop errors: {len(rep)} across {rep['language_code'].nunique()} languages")

# Multi-service languages (hardest cases)
multi = (
	rep.groupby(["language_code", "language_name"])
	.agg(n_services=("service", "nunique"), services=("service", lambda x: sorted(x.unique())))
	.reset_index()
	.query("n_services >= 2")
	.sort_values("n_services", ascending=False)
)
print(f"Languages flagged by ≥2 services ({len(multi)}):")
for _, r in multi.iterrows():
	print(f"  {r['language_code']} ({r['language_name']}): {r['services']}")

# Cross-check: do any also have an honest refusal from a *different* service?
print("Honest-refusal cross-service overlap (loop + honest refusal from different service):")
for lang, grp in rep.groupby("language_code"):
	rep_svcs = set(grp["service"])
	honest = llm_err[
		(llm_err["language_code"] == lang) &
		(llm_err["category"].isin(["knowledge_gap", "extinct_ancient"])) &
		(~llm_err["service"].isin(rep_svcs))
	]
	if not honest.empty:
		lang_name = grp["language_name"].iloc[0]
		print(f"  {lang} ({lang_name}): loops in {sorted(rep_svcs)}, "
			  f"honest refusal ({sorted(honest['category'].unique())}) from {sorted(honest['service'].unique())}")

print("Per-service repetition-loop count:")
print(rep["service"].value_counts().to_string())

# Family breakdown (unique languages only)
# language_family is not a column in all_dfs — derive it via get_language_family
fam_map = {
	code: get_language_family(code)
	for code in all_dfs[TARGET_TERMS[0]]["language_code"].unique()
}
rep_unique = rep.drop_duplicates("language_code").copy()
rep_unique["language_family"] = rep_unique["language_code"].map(fam_map)
print("Family breakdown (unique languages):")
print(rep_unique["language_family"].value_counts().to_string())

Repetition-loop errors: 61 across 56 languages
Languages flagged by ≥2 services (3):
  xcr (Carian): ['DeepSeek', 'OpenAI']
  xna (Ancient North Arabian): ['DeepSeek', 'OpenAI']
  zbl (Blissymbols): ['DeepSeek', 'OpenAI']
Honest-refusal cross-service overlap (loop + honest refusal from different service):
  bqv (Koro Wachi): loops in ['DeepSeek'], honest refusal (['knowledge_gap']) from ['OpenAI']
  bsc (bsc): loops in ['DeepSeek'], honest refusal (['knowledge_gap']) from ['Llama', 'OpenAI']
  eka (Ekajuk): loops in ['DeepSeek'], honest refusal (['knowledge_gap']) from ['OpenAI']
  gld (Nanai): loops in ['Gemma'], honest refusal (['knowledge_gap']) from ['Llama', 'OpenAI']
  hit (Hittite): loops in ['OpenAI'], honest refusal (['extinct_ancient', 'knowledge_gap']) from ['Llama']
  kro (kro): loops in ['DeepSeek'], honest refusal (['knowledge_gap']) from ['OpenAI']
  lab (Linear A): loops in ['Gemma'], honest refusal (['extinct_ancient']) from ['Gemini', 'OpenAI']
  lcp (Western Lawa): l

#### Refusal Grammar

Some services refuse in highly formulaic ways. OpenAI is the clearest case in this run: its generic-refusal and knowledge-gap errors often begin with a small set of repeated opening phrases. The code cell below calculates the current refusal counts and phrase distribution live from the error logs.

This uniformity matters for disagreement analysis: two services "disagreeing" may in reality be one refusing and one hallucinating — a failure of different kinds, not a genuine dispute about the translation.


In [11]:
# OpenAI refusal grammar analysis
oa = llm_err[llm_err["service"] == "OpenAI"]
refusals = oa[oa["category"].isin(["generic_refusal", "knowledge_gap"])].copy()

print(f"OpenAI refusal errors: {len(refusals)} ({len(refusals[refusals['category']=='generic_refusal'])} generic + {len(refusals[refusals['category']=='knowledge_gap'])} knowledge_gap)")
print()

openings = refusals["clean_msg"].str.split().str[:8].str.join(" ")
top = openings.value_counts()
print("Top opening phrases (first 8 words):")
cumulative = 0
for i, (phrase, n) in enumerate(top.head(8).items()):
    pct = n / len(refusals) * 100
    cumulative += n
    cum_pct = cumulative / len(refusals) * 100
    print(f"  {n:3d} ({pct:5.1f}% | cum {cum_pct:5.1f}%) '{phrase}'")

top5_n = top.head(5).sum()
print(f"\nTop 5 openings cover {top5_n}/{len(refusals)} = {top5_n/len(refusals)*100:.1f}% of OpenAI refusals")

# Compare with other services' refusal patterns
print("\nComparison: refusal openings by service")
for svc in sorted(llm_err["service"].unique()):
    if svc == "OpenAI":
        continue
    svc_ref = llm_err[(llm_err["service"] == svc) & (llm_err["category"].isin(["generic_refusal", "knowledge_gap"]))]
    if len(svc_ref) == 0:
        continue
    print(f"  {svc} ({len(svc_ref)} refusals):")
    svc_openings = svc_ref["clean_msg"].str.split().str[:8].str.join(" ").value_counts()
    for phrase, n in svc_openings.head(3).items():
        print(f"    {n:3d}  '{phrase}'")


OpenAI refusal errors: 201 (115 generic + 86 knowledge_gap)

Top opening phrases (first 8 words):
  111 ( 55.2% | cum  55.2%) 'I'm sorry, but I can't provide a translation'
   66 ( 32.8% | cum  88.1%) 'I'm sorry, but I currently do not have'
    9 (  4.5% | cum  92.5%) 'I'm sorry, but I cannot provide a translation'
    5 (  2.5% | cum  95.0%) 'I'm sorry, but I don't have the capability'
    3 (  1.5% | cum  96.5%) 'I'm sorry, but as of my last update,'
    3 (  1.5% | cum  98.0%) 'I'm sorry, but I can't assist with that'
    2 (  1.0% | cum  99.0%) 'I'm sorry, but I currently don't have the'
    1 (  0.5% | cum  99.5%) 'Kutenai (ISO kut) is a language spoken by'

Top 5 openings cover 194/201 = 96.5% of OpenAI refusals

Comparison: refusal openings by service
  Gemini (11 refusals):
      2  'I cannot provide a translation of "Digital Humanities"'
      2  'I am unable to provide a translation of'
      1  'I cannot provide a translation into "xlc" because'
  Llama (87 refusals):
     

## 2.2 Data Quality and Curation

Three interlocking quality signals — missing rationales, script anomalies, and script disagreements — are surfaced here and consolidated into a per-language quality-flags file used by the review explorer.

### Missing Rationales

LLM services are expected to return both a translated term *and* a rationale explaining their choice. Two failure modes arise in practice:

- **Missing rationale** — the service returned a translation but left the rationale field null or empty.
- **Placeholder rationale** — the model returned a literal string such as `"No rationale provided"` instead of real reasoning.

Both are treated as equivalent to a missing rationale. `load_variant_df`(`scripts/exploration/explore_confidence_within_variant.py`) calls `enforce_translation_rationale_pairing` from `scripts/utils.py` before returning, which nulls out:
- any translation whose rationale is absent or a placeholder, and
- any rationale (or placeholder) whose translation is absent.

This means every notebook and script that calls `load_variant_df` — including this one — already receives clean, paired data. `explore_disagreements.py` additionally drops any LLM service translation that has no real rationale in the loaded variant file, so unpaired rows cannot influence the disagreement classifier.

The table below counts mismatches *before* pairing enforcement, showing the raw scope of the problem per service × prompt variant.

In [12]:
LLM_RAT_COLS = {
    "Claude":   "claude_translation_rationale",
    "OpenAI":   "openai_translation_rationale",
    "Gemini":   "gemini_translation_rationale",
    "DeepSeek": "deepseek_translation_rationale",
    "Llama":    "llama_translation_rationale",
    "Gemma":    "gemma_translation_rationale",
    "Qwen":     "qwen_translation_rationale",
    "Mistral":  "mistral_translation_rationale",
}

mismatch_rows = []
df = all_dfs[TARGET_TERMS[0]]

for variant in VARIANTS:
    vdf = df[df["prompt_variant"] == variant]
    for service, trans_col in LLM_SERVICES.items():
        rat_col = LLM_RAT_COLS.get(service)
        if trans_col not in vdf.columns or rat_col not in vdf.columns:
            continue
        has_trans = vdf[trans_col].notna() & ~vdf[trans_col].astype(str).str.strip().isin(["", "nan"])
        has_rat   = vdf[rat_col].notna()   & ~vdf[rat_col].astype(str).str.strip().isin(["", "nan"])
        t_no_r = int((has_trans & ~has_rat).sum())
        r_no_t = int((~has_trans & has_rat).sum())
        mismatch_rows.append({
            "service":             service,
            "variant":             variant,
            "n_translation":       int(has_trans.sum()),
            "n_rationale":         int(has_rat.sum()),
            "trans_no_rationale":  t_no_r,
            "rationale_no_trans":  r_no_t,
        })

mismatch_df = pd.DataFrame(mismatch_rows)

def highlight_nonzero(val):
    return "background-color: #ffe0e0" if isinstance(val, int) and val > 0 else ""

display(
    mismatch_df.style
    .map(highlight_nonzero, subset=["trans_no_rationale", "rationale_no_trans"])
    .format({
        "n_translation":      "{:,}",
        "n_rationale":        "{:,}",
        "trans_no_rationale": "{:,}",
        "rationale_no_trans": "{:,}",
    })
    .set_caption("LLM service translation ↔ rationale mismatches (red = non-zero; these rows are excluded from explore_disagreements.py)")
)

,service,variant,n_translation,n_rationale,trans_no_rationale,rationale_no_trans
0,OpenAI,minimal,797,797,0,0
1,Claude,minimal,881,881,0,0
2,Gemini,minimal,846,848,0,2
3,DeepSeek,minimal,868,870,0,2
4,Llama,minimal,787,787,0,0
5,Gemma,minimal,865,865,0,0
6,Qwen,minimal,875,877,0,2
7,Mistral,minimal,829,829,1,1
8,OpenAI,fluent_speaker,631,750,0,119
9,Claude,fluent_speaker,881,881,0,0


### Script and Directionality

RTL languages and CJK-script languages are structurally harder for translation pipelines. Does coverage drop for these groups?

In [13]:
# Build script-group sets from the comprehensive language metadata
_lang_meta = read_csv_file(os.path.join(DATA_DIR, "metadata_files", "language_codes_comprehensive.csv"))
_lang_meta = _lang_meta.dropna(subset=["language_code"])

# RTL: CLDR-derived directionality column (78 codes); already has FORCE_LTR overrides applied
# — more complete than any hardcoded list and correctly excludes diq/ha/ku/uz/uz_AF
RTL_CODES = set(_lang_meta.loc[_lang_meta["directionality"] == "rtl", "language_code"])

# CJK / SE Asian: derive from primary_script
CJK_SCRIPTS = {
    "Bopomofo", "Simplified", "Traditional", "Japanese", "Korean", "Katakana",
    "Tibetan", "Myanmar", "Khmer", "Lao", "Thai", "Han",
}
cjk_from_csv = set(_lang_meta.loc[_lang_meta["primary_script"].isin(CJK_SCRIPTS), "language_code"].dropna())
# Chinese variant codes present in pipeline data but missing primary_script in the metadata CSV
CJK_EXTRA = {"zh-tw", "zh-classical", "zh-min-nan", "zh-yue", "cdo"}
CJK_CODES = cjk_from_csv | CJK_EXTRA

print(f"RTL codes: {len(RTL_CODES)}  |  CJK/SE Asian codes: {len(CJK_CODES)}")

def script_group(code):
    if code in RTL_CODES:
        return "RTL"
    if code in CJK_CODES:
        return "CJK / SE Asian"
    return "LTR (Latin/other)"

term = TARGET_TERMS[0]
baseline = all_dfs[term][all_dfs[term]["prompt_variant"] == "minimal"].copy()
svc_cols = {s: c for s, c in HEATMAP_SERVICES.items() if c in baseline.columns}
baseline["n_services"] = baseline[[c for c in svc_cols.values()]].notna().sum(axis=1)
baseline["script"] = baseline["language_code"].apply(script_group)

script_rows = []
for service, col in svc_cols.items():
    for script, grp in baseline.groupby("script"):
        n = int(grp[col].notna().sum())
        total = len(grp)
        script_rows.append({"service": service, "script": script, "n": n, "total": total})
script_df = pd.DataFrame(script_rows)

script_order = ["LTR (Latin/other)", "RTL", "CJK / SE Asian"]
svc_order = list(HEATMAP_SERVICES.keys())
max_n_script = script_df["n"].max()

heatmap = alt.Chart(script_df).mark_rect().encode(
    x=alt.X("service:N", sort=svc_order, title=None),
    y=alt.Y("script:N", sort=script_order, title=None),
    color=alt.Color("n:Q",
        scale=alt.Scale(scheme="yellowgreen", domain=[0, max_n_script]),
        title="languages translated"),
    tooltip=["service:N","script:N",
             alt.Tooltip("n:Q",title="translated"),
             alt.Tooltip("total:Q",title="total in group")],
)
hmap_text = heatmap.mark_text(fontSize=10).encode(
    text="n:Q",
    color=alt.condition(alt.datum.n > max_n_script * 0.6, alt.value("white"), alt.value("black")),
)

box = alt.Chart(baseline).mark_boxplot(extent="min-max").encode(
    x=alt.X("script:N", sort=script_order, title=None),
    y=alt.Y("n_services:Q", title="services that translated it", scale=alt.Scale(domain=[0,9])),
    color=alt.Color("script:N", sort=script_order, legend=None),
    tooltip=["script:N"],
).properties(width=260, height=220, title="Coverage depth by script group")

(heatmap + hmap_text).properties(
    width=480, height=100,
    title="Languages translated per service by script group",
) & box

RTL codes: 79  |  CJK/SE Asian codes: 12


alt.VConcatChart(...)

#### Script Agreement Across LLM Services

How often do the eight LLM services agree on *which script* to use for a given language? Full agreement is the norm for well-resourced languages, but disagreement reveals where models are uncertain about the target orthography — or are silently romanising rather than generating the target script.

The second half of this section compares local Ollama models (Llama, Gemma, Qwen, Mistral) against API models (Claude, OpenAI, Gemini, DeepSeek) to see which category is more often the source of script disagreements.

In [14]:
from collections import Counter
from scripts.utils import detect_dominant_script

LLM_SCRIPT_COLS = {
    'Claude':   'claude_translated_term',
    'OpenAI':   'openai_translated_term',
    'Gemini':   'gemini_translated_term',
    'DeepSeek': 'deepseek_translated_term',
    'Llama':    'llama_translated_term',
    'Gemma':    'gemma_translated_term',
    'Qwen':     'qwen_translated_term',
    'Mistral':  'mistral_translated_term',
}

df = all_dfs[TARGET_TERMS[0]]

# ── Part 1: general script agreement across all 8 LLMs ───────────────────────
agreement_rows = []
outlier_rows = []

for variant in VARIANTS:
    vdf = df[df['prompt_variant'] == variant]
    for _, row in vdf.iterrows():
        lang = row['language_code']
        fam  = get_language_family(lang)
        scripts = {}
        for svc, col in LLM_SCRIPT_COLS.items():
            val = row.get(col)
            if pd.notna(val) and str(val).strip() not in ('', 'nan'):
                s = detect_dominant_script(str(val))
                if s != 'Unknown':
                    scripts[svc] = s
        if len(scripts) < 2:
            continue

        n_distinct = len(set(scripts.values()))
        agreement_rows.append({
            'language_code':     lang,
            'language_name':     row.get('language_name', lang),
            'language_family':   fam,
            'variant':           variant,
            'n_services':        len(scripts),
            'n_distinct_scripts': n_distinct,
        })

        if n_distinct > 1:
            majority_script = Counter(scripts.values()).most_common(1)[0][0]
            for svc, script in scripts.items():
                if script != majority_script:
                    outlier_rows.append({
                        'language_code':   lang,
                        'language_family': fam,
                        'variant':         variant,
                        'service':         svc,
                        'majority_script': majority_script,
                        'outlier_script':  script,
                    })

agree_df   = pd.DataFrame(agreement_rows)
outlier_df = pd.DataFrame(outlier_rows)

n_rows     = len(agree_df)
n_disagree = (agree_df['n_distinct_scripts'] > 1).sum()
print(f"Rows with ≥2 LLM translations: {n_rows:,}")
print(f"Any script disagreement: {n_disagree:,} ({n_disagree/n_rows*100:.1f}%)")
print(f"Full agreement (all same script): {(agree_df['n_distinct_scripts']==1).sum():,} "
      f"({(agree_df['n_distinct_scripts']==1).mean()*100:.1f}%)")

# Chart 1: distribution of n_distinct_scripts per row
label_map = {1: '1 — all agree', 2: '2 scripts', 3: '3 scripts', 4: '4 scripts',
             5: '5 scripts', 6: '6 scripts', 7: '7 scripts', 8: '8 scripts'}
dist = (
    agree_df['n_distinct_scripts'].value_counts().reset_index()
    .rename(columns={'n_distinct_scripts': 'n_distinct', 'count': 'count'})
)
dist['pct']   = (dist['count'] / n_rows * 100).round(1)
dist['label'] = dist['n_distinct'].map(label_map)
label_order   = [label_map[i] for i in sorted(label_map) if i in dist['n_distinct'].values]

dist_bar = alt.Chart(dist).mark_bar().encode(
    x=alt.X('label:N', sort=label_order, title=None, axis=alt.Axis(labelAngle=0)),
    y=alt.Y('count:Q', title='rows (language × variant)'),
    color=alt.condition(
        alt.datum.n_distinct == 1, alt.value('#388e3c'), alt.value('#d32f2f')),
    tooltip=['label:N', 'count:Q', alt.Tooltip('pct:Q', format='.1f', title='%')],
).properties(width=300, height=220, title='Script agreement across 8 LLM services')
dist_text = dist_bar.mark_text(dy=-6, fontSize=11).encode(
    text=alt.Text('pct:Q', format='.1f'), color=alt.value('black'))

# Chart 2: which service is most often the outlier?
svc_outlier = (
    outlier_df.groupby('service').size().reset_index(name='n')
    .sort_values('n', ascending=False)
)
svc_outlier['pct'] = (svc_outlier['n'] / len(outlier_df) * 100).round(1)

outlier_bar = alt.Chart(svc_outlier).mark_bar().encode(
    y=alt.Y('service:N', sort=alt.EncodingSortField('n', order='descending'), title=None),
    x=alt.X('n:Q', title='times as script outlier'),
    color=alt.Color('service:N', legend=None, scale=alt.Scale(scheme='tableau10')),
    tooltip=['service:N', 'n:Q', alt.Tooltip('pct:Q', format='.1f', title='% of all outlier events')],
).properties(width=300, height=200, title='Which service is most often the script outlier?')
outlier_text = outlier_bar.mark_text(align='left', dx=4, fontSize=9).encode(
    text='n:Q', color=alt.value('black'))

# Chart 3: families with most script disagreement
fam_disagree = (
    agree_df[agree_df['n_distinct_scripts'] > 1]
    .groupby('language_family')['language_code']
    .nunique().reset_index(name='n_langs')
    .sort_values('n_langs', ascending=False).head(12)
)
fam_dis_bar = alt.Chart(fam_disagree).mark_bar(color='#1976d2').encode(
    y=alt.Y('language_family:N',
            sort=alt.EncodingSortField('n_langs', order='descending'), title=None),
    x=alt.X('n_langs:Q', title='unique languages with any script disagreement'),
    tooltip=['language_family:N', 'n_langs:Q'],
).properties(width=300, height=260, title='Families with most script disagreement')
fam_dis_text = fam_dis_bar.mark_text(align='left', dx=4, fontSize=9).encode(
    text='n_langs:Q', color=alt.value('black'))

(dist_bar + dist_text).display()
((outlier_bar + outlier_text) | (fam_dis_bar + fam_dis_text)).display()

# ── Part 2: Local vs API model script outlier analysis ───────────────────────
LOCAL_MODELS = ['Llama', 'Gemma', 'Qwen', 'Mistral']
API_MODELS   = ['Claude', 'OpenAI', 'Gemini', 'DeepSeek']

local_script_rows = []

for variant in VARIANTS:
    vdf = df[df['prompt_variant'] == variant]
    for _, row in vdf.iterrows():
        scripts = {}
        for svc, col in LLM_SCRIPT_COLS.items():
            val = row.get(col)
            if pd.notna(val) and str(val).strip() not in ('', 'nan'):
                s = detect_dominant_script(str(val))
                if s != 'Unknown':
                    scripts[svc] = s

        api_scripts = {s: scripts[s] for s in API_MODELS if s in scripts}
        if len(api_scripts) < 2:
            continue
        majority_script = Counter(api_scripts.values()).most_common(1)[0][0]
        majority_count  = list(api_scripts.values()).count(majority_script)
        if majority_count < 2:
            continue

        for local_svc in LOCAL_MODELS:
            if local_svc not in scripts:
                continue
            if scripts[local_svc] != majority_script:
                local_script_rows.append({
                    'variant':         variant,
                    'language_code':   row['language_code'],
                    'language_name':   row.get('language_name', row['language_code']),
                    'language_family': get_language_family(row['language_code']),
                    'local_service':   local_svc,
                    'local_script':    scripts[local_svc],
                    'majority_script': majority_script,
                    'api_agree_n':     majority_count,
                })

local_sd = pd.DataFrame(local_script_rows)

print(f"\nLocal model script outliers (API majority ≥2 agree on different script): "
      f"{len(local_sd)} rows, {local_sd['language_code'].nunique() if len(local_sd) else 0} unique languages")

if len(local_sd) > 0:
    # Chart 4: which local model is most often the script outlier?
    svc_counts = (
        local_sd.groupby('local_service').size().reset_index(name='n')
        .sort_values('n', ascending=False)
    )
    svc_counts['pct'] = (svc_counts['n'] / len(local_sd) * 100).round(1)

    svc_bar = alt.Chart(svc_counts).mark_bar().encode(
        y=alt.Y('local_service:N', sort=alt.EncodingSortField('n', order='descending'), title=None),
        x=alt.X('n:Q', title='times as script outlier vs API majority'),
        color=alt.Color('local_service:N', legend=None, scale=alt.Scale(scheme='tableau10')),
        tooltip=['local_service:N', 'n:Q', alt.Tooltip('pct:Q', format='.1f', title='% of local outlier events')],
    ).properties(width=300, height=160, title='Which local model deviates most from API script consensus?')
    svc_text = svc_bar.mark_text(align='left', dx=4, fontSize=9).encode(
        text='n:Q', color=alt.value('black'))

    # Chart 5: script-pair breakdown (API majority → local model's script)
    pair_counts = (
        local_sd.groupby(['majority_script', 'local_script'])
        .size().reset_index(name='n')
        .sort_values('n', ascending=False).head(12)
    )
    pair_counts['pair'] = pair_counts['majority_script'] + ' → ' + pair_counts['local_script']

    pair_bar = alt.Chart(pair_counts).mark_bar().encode(
        y=alt.Y('pair:N', sort='-x', title=None),
        x=alt.X('n:Q', title='occurrences (across all variants)'),
        color=alt.Color('local_script:N', title="local model's script",
                        scale=alt.Scale(scheme='tableau10')),
        tooltip=['pair:N', 'n:Q', 'majority_script:N', 'local_script:N'],
    ).properties(width=380, height=280,
                 title="Local model script mismatches: API majority script → local model's script")
    pair_text = pair_bar.mark_text(align='left', dx=4, fontSize=9).encode(
        text='n:Q', color=alt.value('black'))

    # Chart 6: top families in local model outlier cases
    fam_local = (
        local_sd.groupby('language_family')['language_code']
        .nunique().reset_index(name='n_langs')
        .sort_values('n_langs', ascending=False).head(10)
    )
    fam_loc_bar = alt.Chart(fam_local).mark_bar(color='#8b00d4').encode(
        y=alt.Y('language_family:N',
                sort=alt.EncodingSortField('n_langs', order='descending'), title=None),
        x=alt.X('n_langs:Q', title='unique languages'),
        tooltip=['language_family:N', 'n_langs:Q'],
    ).properties(width=280, height=240, title='Families most affected by local model script switching')
    fam_loc_text = fam_loc_bar.mark_text(align='left', dx=4, fontSize=9).encode(
        text='n_langs:Q', color=alt.value('black'))

    (svc_bar + svc_text).display()
    (pair_bar + pair_text).display()
    (fam_loc_bar + fam_loc_text).display()
else:
    print("No local model script outlier cases found.")


Rows with ≥2 LLM translations: 3,524
Any script disagreement: 1,053 (29.9%)
Full agreement (all same script): 2,471 (70.1%)


alt.LayerChart(...)

alt.HConcatChart(...)


Local model script outliers (API majority ≥2 agree on different script): 1851 rows, 366 unique languages


alt.LayerChart(...)

alt.LayerChart(...)

alt.LayerChart(...)

#### Script Disagreement by Language Family

The overall script-disagreement rate is not uniform across language families. The code cell below calculates the current rate and family distribution live from the script-agreement table.

Two patterns are worth flagging:

**High rate, low expected script count** — families where most languages have a single established orthography but models still produce script variation. This is model-injected noise rather than genuine linguistic ambiguity.

**High expected script count, no disagreement** — languages with multiple registered scripts where all LLMs nonetheless agree on a single script. These are typically languages with one dominant script in digital text, so model convergence can be correct rather than coincidental.


In [15]:
# Script disagreement by language family
# agree_df and outlier_df are built in the cell above (Script Agreement Across LLM Services)
# _lang_meta is built in the Script and Directionality setup cell

scripts_per_lang = (
    _lang_meta[["language_code", "n_scripts"]]
    .dropna(subset=["language_code"])
    .copy()
)

# Summarize at language level: disagreement = any variant had n_distinct_scripts > 1
lang_disagr = (
    agree_df.groupby(["language_code", "language_family"])
    .agg(has_script_disagr=("n_distinct_scripts", lambda x: (x > 1).any()))
    .reset_index()
)
lang_disagr = lang_disagr.merge(scripts_per_lang, on="language_code", how="left")

fam_stats = (
    lang_disagr.groupby("language_family")
    .agg(
        n_langs=("language_code", "nunique"),
        n_disagr=("has_script_disagr", "sum"),
        avg_n_scripts=("n_scripts", "mean"),
    )
    .reset_index()
)
fam_stats["rate"] = fam_stats["n_disagr"] / fam_stats["n_langs"]
fam_stats = fam_stats.sort_values("rate", ascending=False)

chart = alt.Chart(fam_stats[fam_stats["n_langs"] >= 5]).mark_bar().encode(
    y=alt.Y("language_family:N", sort="-x", title=None),
    x=alt.X("rate:Q", axis=alt.Axis(format="%"), title="script disagreement rate"),
    color=alt.Color("avg_n_scripts:Q",
                    scale=alt.Scale(scheme="blues", domainMin=1),
                    title="avg registered scripts"),
    tooltip=[
        "language_family:N",
        alt.Tooltip("rate:Q", format=".1%"),
        alt.Tooltip("n_langs:Q", title="languages"),
        alt.Tooltip("n_disagr:Q", title="disagreements"),
        alt.Tooltip("avg_n_scripts:Q", format=".2f", title="avg scripts"),
    ],
).properties(width=420, height=280,
             title="Script disagreement rate by family (n ≥ 5 languages)")
display(chart)

# Drill into top 3 high-rate / low avg_n_scripts families (model-injected noise)
high_rate = fam_stats[
    (fam_stats["n_langs"] >= 5) &
    (fam_stats["rate"] > 0.5) &
    (fam_stats["avg_n_scripts"] < 1.3)
].head(3)

print("High script-disagreement rate, low expected script diversity (model-injected noise):")
for _, frow in high_rate.iterrows():
    fam = frow["language_family"]
    print(f"\n  {fam}  rate={frow['rate']:.0%}, {int(frow['n_langs'])} langs, "
          f"avg {frow['avg_n_scripts']:.2f} scripts/lang")
    # Outlier services for this family
    fam_out = (
        outlier_df[outlier_df["language_family"] == fam]
        .groupby("service")["language_code"]
        .nunique()
        .sort_values(ascending=False)
    )
    print(f"  Outlier services: {dict(fam_out)}")
    # Example languages
    ex_langs = (
        lang_disagr[
            (lang_disagr["language_family"] == fam) &
            (lang_disagr["has_script_disagr"])
        ][["language_code"]]
        .drop_duplicates()
        .merge(
            all_dfs[TARGET_TERMS[0]][["language_code", "language_name"]].drop_duplicates("language_code"),
            on="language_code", how="left"
        )
        .head(5)
    )
    for _, r in ex_langs.iterrows():
        print(f"    {r['language_code']} ({r['language_name']})")

# Languages with n_scripts > 1 but no disagreement (model correctly converges)
multi_nodisr = lang_disagr[
    (lang_disagr["n_scripts"] > 1) & (~lang_disagr["has_script_disagr"])
]
print(f"\nLanguages with ≥2 registered scripts but no cross-service disagreement: {len(multi_nodisr)}")
print("(Models converge on dominant digital-text script despite multiple registered orthographies)")


alt.Chart(...)

High script-disagreement rate, low expected script diversity (model-injected noise):

  Tai-Kadai languages  rate=100%, 10 langs, avg 1.10 scripts/lang
  Outlier services: {'Gemma': np.int64(8), 'Llama': np.int64(8), 'Mistral': np.int64(5), 'Claude': np.int64(4), 'Gemini': np.int64(4), 'OpenAI': np.int64(4), 'DeepSeek': np.int64(3), 'Qwen': np.int64(2)}
    blt (Tai Dam)
    khb (Lü)
    lo (Lao)
    nod (Northern Thai)
    shn (Shan)

  Sino-Tibetan languages  rate=92%, 48 langs, avg 1.19 scripts/lang
  Outlier services: {'Gemma': np.int64(38), 'Llama': np.int64(35), 'Mistral': np.int64(22), 'Qwen': np.int64(22), 'Gemini': np.int64(12), 'DeepSeek': np.int64(11), 'OpenAI': np.int64(11), 'Claude': np.int64(8)}
    bap (Bantawa)
    bft (Balti)
    bo (Tibetan)
    brx (Boro)
    cdo (Min Dong Chinese)

  Eskimo-Aleut languages  rate=80%, 5 langs, avg 1.00 scripts/lang
  Outlier services: {'Mistral': np.int64(4), 'Gemini': np.int64(3), 'Gemma': np.int64(3), 'Qwen': np.int64(3), 'OpenAI':

#### Mixed-Script Translations

A subtler script anomaly: individual translations that contain characters from two or more distinct scripts within a single string. For example, a Cyrillic-target language that receives "Цифровые Humanities" mixes Cyrillic and Latin in one output. This can indicate a model giving up mid-word, defaulting to an English fragment, or failing to transliterate a technical term.

Detection rule: a translation is "mixed" when a secondary script accounts for ≥ 10 % of all script-significant characters. CJK and Japanese syllabaries are treated as one family (Hiragana/Katakana + Kanji co-occurrence is normal). Prompt-invariant services (Wikipedia community translation, Google Translate, EasyNMT, Lingvanex) use the minimal-variant row only.

In [16]:
import unicodedata as ud
from scripts.utils import char_script

MIXED_THRESHOLD = 0.10  # secondary script must be ≥10% of script chars to count

def script_profile(text):
    """Return {script: char_count}, ignoring punctuation/digits/spaces."""
    if not isinstance(text, str) or not text.strip():
        return {}
    counts = Counter()
    for ch in text:
        cp = ord(ch)
        cat = ud.category(ch)
        if cat[0] in ('Z', 'P', 'S', 'C') or cat == 'Nd':
            continue
        s = char_script(cp)
        if s == 'Other':
            continue
        # CJK kanji + Japanese kana co-occur normally in Japanese — merge them
        s = 'CJK/Japanese' if s in ('CJK', 'Japanese') else s
        counts[s] += 1
    return dict(counts)

def _is_mixed(profile, threshold=MIXED_THRESHOLD):
    if len(profile) < 2:
        return False
    total = sum(profile.values())
    dominant = max(profile.values())
    return total > 0 and (total - dominant) / total >= threshold

# ── Build mixed-script rows across all services and variants ─────────────────
ALL_SERVICES = {**BASELINE_SERVICES, **LLM_SERVICES}
term = TARGET_TERMS[0]
df_all = all_dfs[term].copy()
df_all["language_family"] = df_all["language_code"].apply(get_language_family)

# For prompt-invariant services use minimal variant only (they do not vary by prompt)
prompt_invariant_rows_df = df_all[df_all["prompt_variant"] == "minimal"].copy()

mixed_rows = []
for service, col in ALL_SERVICES.items():
    if col not in df_all.columns:
        continue
    src = prompt_invariant_rows_df if service in PROMPT_INVARIANT_SERVICES else df_all
    for _, row in src.iterrows():
        val = row.get(col)
        if not isinstance(val, str) or not val.strip():
            continue
        profile = script_profile(val)
        if not profile:
            continue
        total_chars = sum(profile.values())
        scripts_present = sorted(profile, key=lambda s: -profile[s])
        dominant = scripts_present[0]
        minority_frac = (total_chars - profile[dominant]) / total_chars
        mixed_rows.append({
            "service":         service,
            "language_code":   row["language_code"],
            "language_name":   row.get("language_name", row["language_code"]),
            "language_family": row["language_family"],
            "variant":         row["prompt_variant"],
            "translation":     val,
            "dominant_script": dominant,
            "n_scripts":       len(profile),
            "minority_frac":   round(minority_frac, 3),
            "is_mixed":        _is_mixed(profile),
            "scripts_str":     " + ".join(scripts_present),
        })

mixed_df = pd.DataFrame(mixed_rows)

print(f"Total translations checked: {len(mixed_df):,}")
print(f"Mixed-script ({MIXED_THRESHOLD*100:.0f}% threshold): {mixed_df['is_mixed'].sum():,} "
      f"({mixed_df['is_mixed'].mean()*100:.1f}%)")
print()
print(mixed_df[mixed_df['is_mixed']].groupby('service')['is_mixed'].sum().sort_values(ascending=False).to_string())

# ── Chart 1: % mixed-script by service ───────────────────────────────────────
svc_mix = (
    mixed_df.groupby("service")
    .agg(n_total=("is_mixed", "count"), n_mixed=("is_mixed", "sum"))
    .assign(pct_mixed=lambda d: (d["n_mixed"] / d["n_total"] * 100).round(1))
    .reset_index()
    .sort_values("pct_mixed", ascending=False)
)
svc_bar = alt.Chart(svc_mix).mark_bar().encode(
    y=alt.Y("service:N", sort=alt.EncodingSortField("pct_mixed", order="descending"), title=None),
    x=alt.X("pct_mixed:Q", title="% translations with mixed script"),
    color=alt.Color("service:N", legend=None, scale=alt.Scale(scheme="tableau10")),
    tooltip=["service:N",
             alt.Tooltip("pct_mixed:Q", format=".1f", title="% mixed"),
             alt.Tooltip("n_mixed:Q", title="mixed count"),
             alt.Tooltip("n_total:Q", title="total")],
).properties(width=340, height=220, title="Mixed-script translations by service")
svc_text = svc_bar.mark_text(align="left", dx=4, fontSize=9).encode(
    text=alt.Text("pct_mixed:Q", format=".1f"), color=alt.value("black"))

# ── Chart 2: dominant script pairs (what gets mixed with what) ───────────────
only_mixed = mixed_df[mixed_df["is_mixed"]].copy()
pair_counts = (
    only_mixed.groupby(["scripts_str", "service"])
    .size().reset_index(name="n")
    .sort_values("n", ascending=False)
)
top_pairs = pair_counts.groupby("scripts_str")["n"].sum().nlargest(10).index
pair_top = pair_counts[pair_counts["scripts_str"].isin(top_pairs)]

pair_bar = alt.Chart(pair_top).mark_bar().encode(
    y=alt.Y("scripts_str:N", sort=alt.EncodingSortField("n", order="descending"), title=None),
    x=alt.X("n:Q", title="occurrences"),
    color=alt.Color("service:N", scale=alt.Scale(scheme="tableau10"), title="Service"),
    tooltip=["scripts_str:N", "service:N", "n:Q"],
).properties(width=380, height=240, title="Most common script mixtures (top 10 pairs)")

# ── Chart 3: affected language families ──────────────────────────────────────
fam_mix = (
    only_mixed.groupby("language_family")["language_code"]
    .nunique().reset_index(name="n_langs")
    .sort_values("n_langs", ascending=False).head(12)
)
fam_bar = alt.Chart(fam_mix).mark_bar(color="#6a1b9a").encode(
    y=alt.Y("language_family:N", sort=alt.EncodingSortField("n_langs", order="descending"), title=None),
    x=alt.X("n_langs:Q", title="unique languages with mixed-script output"),
    tooltip=["language_family:N", "n_langs:Q"],
).properties(width=300, height=240, title="Language families with most mixed-script languages")
fam_text = fam_bar.mark_text(align="left", dx=4, fontSize=9).encode(
    text="n_langs:Q", color=alt.value("black"))

(svc_bar + svc_text).display()
(pair_bar | (fam_bar + fam_text)).display()

# ── Sample mixed-script translations ─────────────────────────────────────────
print("\nSample mixed-script translations (≥25% minority script):")
samples = (
    only_mixed[only_mixed["minority_frac"] >= 0.25]
    [["service","language_name","dominant_script","scripts_str","minority_frac","translation"]]
    .sort_values(["service","minority_frac"], ascending=[True, False])
    .groupby("service").head(3)
    .reset_index(drop=True)
)
pd.set_option("display.max_colwidth", 60)
display(samples)

Total translations checked: 27,364
Mixed-script (10% threshold): 374 (1.4%)

service
Gemma               153
Qwen                 98
Llama                56
Mistral              37
Claude               14
Gemini               13
DeepSeek              2
Google Translate      1


alt.LayerChart(...)

alt.HConcatChart(...)


Sample mixed-script translations (≥25% minority script):


,service,language_name,dominant_script,scripts_str,minority_frac,translation
0,Claude,mey,Latin,Latin + Arabic,0.449,al-ʿulūm al-insāniyya ar-raqmiyya / العلوم الإنسانية الر...
1,Claude,Ancient North Arabian,Latin,Latin + Arabic,0.286,ḥkmت ṣnʿت
2,Claude,Sogdian,Latin,Latin + Greek,0.250,δβ'nyk 'nš'n-δ'nšn'
3,DeepSeek,Sogdian,Latin,Latin + Greek,0.250,δβ'nyk 'nš'n-δ'nšn'
4,Gemini,Cherokee,Cherokee,Cherokee + Korean,0.400,Ꮧ지털 ᏴᏫ
5,Gemini,Lisu,Myanmar,Myanmar + Lisu,0.333,နီꓫလီꓸ မူꓲဖီꓲ လီꓲယိꓲ
6,Gemini,Cherokee,Cherokee,Cherokee + Korean,0.286,Ꮧ지털 ᎭᎹᏂᏘ
7,Gemma,Pāli,Thai,Thai + Devanagari + Malayalam,0.562,ดิจิทัล മാനवीय कला
8,Gemma,Kachin,Latin,Latin + Thai + Myanmar + Khmer + Sinhala,0.515,ඩිជីថល လူမှုศาสตร์ (Di-ji-thal lu-mu-saat)
9,Gemma,Tuvinian,Cyrillic,Cyrillic + Latin,0.500,Дигитал хүннүгэ (Digital hunnuge)


#### Curating Mixed-Script Translations

`curate_translation()` implements six classification outcomes:

- **Strip** (Pattern A): removes `(romanization)` parentheticals and slash-separated suffixes, then verifies the remainder is single-script. The curated term is used going forward.
- **Strip** (Pattern C): removes a source-term prefix before a colon separator (e.g. `"Digital Humanities : कार्यान्वित मानवशास्त्र"` → `कार्यान्वित मानवशास्त्र`). Only applied when the prefix contains no characters from the dominant (native) script, so native text with colons is never truncated.
- **Strip** (Pattern D): removes a source-term prefix separated only by whitespace (e.g. `"Digital Humanities के दिशा कौशल"` → `के दिशा कौशल`). Strips the leading run of tokens that share the script of the first token; only accepted if the remainder is single-script.
- **Strip** (Pattern E): removes a source-term prefix before an equals-sign separator (e.g. `"Digital Humanities = Panagbalikas iti Digital"` → `Panagbalikas iti Digital`). Unlike Patterns C and D, this check runs *before* the mixed-script gate — both sides may share the same script (Latin-script translations), so the string would otherwise pass through unchanged. Only applied when the prefix is entirely Latin.
- **Strip** (Pattern F): removes slash delimiters wrapping the entire term (e.g. `"/Dkawng Thaukhnawng/"` → `"Dkawng Thaukhnawng"`). Distinct from Pattern A's inline slash separator, which strips a suffix; Pattern F handles terms fully enclosed in leading and trailing slashes.
- **Null** (Pattern B): interleaved character noise that stripping cannot fix — treated as a failed translation.

The charts below show how many translations per service are affected, and a sample of before → after transformations.

In [17]:
from scripts.exploration.translation_classifier import curate_translation, curate_df

term = TARGET_TERMS[0]
raw_df = all_dfs[term].copy()

# Apply curation to all *_translated_term columns
curated_df, summary = curate_df(raw_df)

print("Curation summary per service:")
print(summary.to_string(index=False))

# ── Chart: stripped / nulled / placeholder counts by service ──────────────────
summary_long = summary.melt(
    id_vars="service", value_vars=["stripped", "nulled", "placeholder"],
    var_name="action", value_name="count"
)
summary_long = summary_long[summary_long["count"] > 0]

if summary_long.empty:
    print("\nNo mixed-script or placeholder translations found at current threshold.")
else:
    action_bar = alt.Chart(summary_long).mark_bar().encode(
        y=alt.Y("service:N", sort=alt.EncodingSortField("count", order="descending"), title=None),
        x=alt.X("count:Q", title="translations affected"),
        color=alt.Color("action:N",
            scale=alt.Scale(
                domain=["stripped", "nulled", "placeholder"],
                range=["#1976d2", "#d32f2f", "#e67e00"],
            ),
            title="Action"),
        row=alt.Row("action:N", title=None),
        tooltip=["service:N", "action:N", "count:Q"],
    ).properties(width=380, height=120)
    action_bar.display()

# ── Sample before → after table ───────────────────────────────────────────────
ALL_TERM_COLS = {**PROMPT_INVARIANT_SERVICES, **LLM_SERVICES}
sample_rows = []
for service, col in ALL_TERM_COLS.items():
    if col not in raw_df.columns:
        continue
    for (_, raw_row), (_, clean_row) in zip(raw_df.iterrows(), curated_df.iterrows()):
        before = raw_row[col]
        after  = clean_row[col]
        import pandas as _pd
        if not isinstance(before, str):
            continue
        # Only show rows where something changed
        if before == after or (_pd.isna(after) and not isinstance(before, str)):
            continue
        if isinstance(before, str) and (after is None or _pd.isna(after) or before != after):
            _, action = curate_translation(before)
            if action == "unchanged":
                continue
            after_display = str(after) if after is not None and not _pd.isna(after) else f"— {action} —"
            sample_rows.append({
                "service":  service,
                "language": raw_row.get("language_name", raw_row["language_code"]),
                "action":   action,
                "before":   before,
                "after":    after_display,
            })

sample_df = (
    pd.DataFrame(sample_rows)
    .sort_values(["action", "service"])
    .groupby(["action", "service"]).head(2)
    .reset_index(drop=True)
)

if not sample_df.empty:
    print("\nSample transformations:")
    pd.set_option("display.max_colwidth", 70)
    display(sample_df[["service", "language", "action", "before", "after"]])

# ── Coverage impact: how many languages gain/lose coverage after curation ─────
svc_cols = [c for c in ALL_TERM_COLS.values() if c in raw_df.columns]
before_cov = raw_df.groupby("language_code")[svc_cols].agg(lambda x: x.notna().any())
after_cov  = curated_df.groupby("language_code")[svc_cols].agg(lambda x: x.notna().any())

impact_rows = []
for col, service in {v: k for k, v in ALL_TERM_COLS.items() if v in raw_df.columns}.items():
    lost = int((before_cov[col] & ~after_cov[col]).sum())
    impact_rows.append({"service": service, "languages_losing_coverage": lost})

impact_df = pd.DataFrame(impact_rows).sort_values("languages_losing_coverage", ascending=False)
print("\nLanguages losing coverage after nulling/placeholder curation:")
print(impact_df[impact_df["languages_losing_coverage"] > 0].to_string(index=False))


Curation summary per service:
  service  unchanged  stripped  nulled  placeholder
     enmt       3524         0       0            0
wikipedia       3524         0       0            0
       gt       3524         0       0            0
lingvanex       3524         0       0            0
   claude       3518         1       0            5
    llama       3474        19      31            0
    gemma       3395        76      53            0
     qwen       3460        13      51            0
  mistral       3485        27       7            5
   openai       3512         0       0           12
 deepseek       3523         0       0            1
   gemini       3503         3       3           15


alt.Chart(...)


Sample transformations:


,service,language,action,before,after
0,Gemini,Tachelhit,nulled,تودرت ن ⵓⵎⴰⵏ ⴷ ⵓⵎⴰⵏ ⴰⴷⵉⵊⵉⵜⴰⵍ,— nulled —
1,Gemini,Lisu,nulled,နီꓫလီꓸ မူꓲဖီꓲ လီꓲယိꓲ,— nulled —
2,Gemma,Maba,nulled,ඩිජිටල් මානවwissenschaft,— nulled —
3,Gemma,Mongolian,nulled,Дижитал humanistууд,— nulled —
4,Llama,Urdu,nulled,ڈیجیٹل مानवیات,— nulled —
5,Llama,Saraiki,nulled,ڈیجیٹل مानवیات,— nulled —
6,Mistral,Tae',nulled,디지털 umanis,— nulled —
7,Mistral,Abkhaz,nulled,Информационные гumanitарные науки,— nulled —
8,Qwen,Punjabi (Western),nulled,デジタル ਮਨੇਜਮਨ ਵਿਗਿਆਨ,— nulled —
9,Qwen,Russian,nulled,Дigitalnye гуманитарные науки,— nulled —



Languages losing coverage after nulling/placeholder curation:
service  languages_losing_coverage
  Llama                          1


### Automated Review Signals (`automated_review_signals.csv`)

Consolidate the data-quality signals surfaced in §2.2 into a single evidence layer per language. The output file is `automated_review_signals.csv`, which downstream review tools and analysis notebooks use as automated evidence, not as final exclusion authority. The legacy column name `quality_flags` is retained inside the CSV as a compatibility alias, but the section treats these values as review signals rather than final quality judgments.

The signals fall into four interpretive groups:

- **Pipeline integrity**: missing or mismatched translation/rationale pairings, and rationale text in an unexpected language.
- **Output failure**: placeholder/refusal terms, repetition loops, literal Unicode escapes, extreme term length, and unsalvageable mixed-script output.
- **Curation and search complications**: stripped romanization/source wrappers, source-term leakage, language-name pass-through, and very short translations.
- **Analytical signals**: script disagreement and sub-threshold script mixing that should be retained as data unless a later review decision excludes it.

This distinction matters because not every signal means “bad translation.” Some are automatic failure evidence, some are search-readiness problems, and some are features to carry into later analysis.


In [18]:
from collections import defaultdict
from scripts.exploration.translation_classifier import (
    curate_translation, has_source_leakage,
    is_repetition_loop, has_extreme_term_length, has_short_translation, has_unicode_escape,
    is_refusal_rationale, is_transliteration_rationale, is_placeholder_rationale,
    is_language_name_term, has_unexpected_rationale_language, script_mix_detail,
)

term = TARGET_TERMS[0]
df_raw = all_dfs[term].copy()

# ── 1. Missing rationale ─────────────────────────────────────────────────────
# all_dfs uses load_variant_df, which applies enforce_translation_rationale_pairing
# before returning — so mismatches are already erased there. Read the raw
# prompt_services CSVs directly to catch the original pairings.
PLACEHOLDER_RATS = {
    "no rationale provided", "no rationale", "n/a", "none",
    "not applicable", "no explanation provided", "no reason provided",
}

def _has_rat(val):
    if not isinstance(val, str) or not val.strip():
        return False
    return val.strip().rstrip(".").lower() not in PLACEHOLDER_RATS

SVC_FILE_KEY = {
    "Claude": "claude", "OpenAI": "openai", "Gemini": "gemini", "DeepSeek": "deepseek",
    "Llama": "llama", "Gemma": "gemma", "Qwen": "qwen", "Mistral": "mistral",
}
prompt_dir   = os.path.join(DATA_DIR, "translated_terms", term.lower().replace(" ", "_"), "prompt_services")

missing_rat_by_lang = defaultdict(set)
for service, trans_col in LLM_SERVICES.items():
    rat_col  = LLM_RAT_COLS.get(service)
    file_key = SVC_FILE_KEY.get(service)
    if not rat_col or not file_key:
        continue
    for variant in VARIANTS:
        fpath = os.path.join(prompt_dir, f"{file_key}_{variant}_translations.csv")
        if not os.path.exists(fpath):
            continue
        raw_vdf = pd.read_csv(fpath)
        if trans_col not in raw_vdf.columns or rat_col not in raw_vdf.columns:
            continue
        has_trans = raw_vdf[trans_col].notna() & ~raw_vdf[trans_col].astype(str).str.strip().isin(["", "nan"])
        has_rat   = raw_vdf[rat_col].apply(_has_rat)
        mismatch  = (has_trans & ~has_rat) | (~has_trans & has_rat)
        for lc in raw_vdf[mismatch]["language_code"].unique():
            missing_rat_by_lang[lc].add(service)

# ── 1c. Transliteration rationale (model describes a phonetic mapping) ──────
# English-only — fluent_speaker rationales are in the target language and skipped.
transliteration_rat_by_lang = defaultdict(set)
for service, trans_col in LLM_SERVICES.items():
    rat_col  = LLM_RAT_COLS.get(service)
    file_key = SVC_FILE_KEY.get(service)
    if not rat_col or not file_key:
        continue
    for variant in VARIANTS:
        if variant == 'fluent_speaker':
            continue
        fpath = os.path.join(prompt_dir, f'{file_key}_{variant}_translations.csv')
        if not os.path.exists(fpath):
            continue
        raw_vdf = pd.read_csv(fpath)
        if rat_col not in raw_vdf.columns:
            continue
        for _, row in raw_vdf.iterrows():
            rat = row.get(rat_col)
            if isinstance(rat, str) and is_transliteration_rationale(rat):
                transliteration_rat_by_lang[str(row['language_code'])].add(service)

# ── 1b. Refusal rationale (model admits no translation in rationale text) ────
# English-only — fluent_speaker rationales are in the target language and skipped.
refusal_rat_by_lang = defaultdict(set)
for service, trans_col in LLM_SERVICES.items():
    rat_col  = LLM_RAT_COLS.get(service)
    file_key = SVC_FILE_KEY.get(service)
    if not rat_col or not file_key:
        continue
    for variant in VARIANTS:
        if variant == 'fluent_speaker':
            continue
        fpath = os.path.join(prompt_dir, f'{file_key}_{variant}_translations.csv')
        if not os.path.exists(fpath):
            continue
        raw_vdf = pd.read_csv(fpath)
        if rat_col not in raw_vdf.columns:
            continue
        for _, row in raw_vdf.iterrows():
            rat = row.get(rat_col)
            if isinstance(rat, str) and is_refusal_rationale(rat):
                refusal_rat_by_lang[str(row['language_code'])].add(service)

# ── 1d. Placeholder rationale (model admits using a placeholder/stand-in) ───
# English-only — fluent_speaker rationales are in the target language and skipped.
placeholder_rat_by_lang = defaultdict(set)
for service, trans_col in LLM_SERVICES.items():
    rat_col  = LLM_RAT_COLS.get(service)
    file_key = SVC_FILE_KEY.get(service)
    if not rat_col or not file_key:
        continue
    for variant in VARIANTS:
        if variant == 'fluent_speaker':
            continue
        fpath = os.path.join(prompt_dir, f'{file_key}_{variant}_translations.csv')
        if not os.path.exists(fpath):
            continue
        raw_vdf = pd.read_csv(fpath)
        if rat_col not in raw_vdf.columns:
            continue
        for _, row in raw_vdf.iterrows():
            rat = row.get(rat_col)
            if isinstance(rat, str) and is_placeholder_rationale(rat):
                placeholder_rat_by_lang[str(row['language_code'])].add(service)

# ── 1e. Unexpected rationale language (rationale majority non-Latin for English variants) ─
# Skips fluent_speaker (target-language rationales are expected to be non-Latin).
unexp_rat_lang_by_lang = defaultdict(set)
for service, trans_col in LLM_SERVICES.items():
    rat_col  = LLM_RAT_COLS.get(service)
    file_key = SVC_FILE_KEY.get(service)
    if not rat_col or not file_key:
        continue
    for variant in VARIANTS:
        if variant == 'fluent_speaker':
            continue
        fpath = os.path.join(prompt_dir, f'{file_key}_{variant}_translations.csv')
        if not os.path.exists(fpath):
            continue
        raw_vdf = pd.read_csv(fpath)
        if rat_col not in raw_vdf.columns:
            continue
        for _, row in raw_vdf.iterrows():
            rat = row.get(rat_col)
            if isinstance(rat, str) and has_unexpected_rationale_language(rat):
                unexp_rat_lang_by_lang[str(row['language_code'])].add(service)

# ── 2. Mixed script, romanization, and placeholder refusals ──────────────────
ALL_SVC_COLS = {**BASELINE_SERVICES, **LLM_SERVICES}
prompt_invariant_df = df_raw[df_raw["prompt_variant"] == "minimal"].copy()

mixed_by_lang       = defaultdict(set)
roman_by_lang       = defaultdict(set)
placeholder_by_lang = defaultdict(set)  # model explicitly refused to translate

for service, col in ALL_SVC_COLS.items():
    if col not in df_raw.columns:
        continue
    src = prompt_invariant_df if service in PROMPT_INVARIANT_SERVICES else df_raw
    for _, row in src.iterrows():
        val = row.get(col)
        if not isinstance(val, str) or not val.strip():
            continue
        _, action = curate_translation(val)
        lc = row["language_code"]
        if action == "nulled":
            mixed_by_lang[lc].add(service)
        elif action == "stripped":
            roman_by_lang[lc].add(service)
        elif action == "placeholder":
            placeholder_by_lang[lc].add(service)

# ── 3. Script disagreement (from §1.8.1 outlier_df) ──────────────────────────
disagr_by_lang = defaultdict(set)
if not outlier_df.empty:
    for _, row in outlier_df.iterrows():
        disagr_by_lang[row["language_code"]].add(row["service"])

# ── 4. Source term in translation (untranslated leakage) ─────────────────────
# Flags any non-English translation that still contains the English source term
# ("Digital Humanities") or its initials abbreviation ("DH") as a standalone token.
source_term_by_lang = defaultdict(set)
for service, col in ALL_SVC_COLS.items():
    if col not in df_raw.columns:
        continue
    src = prompt_invariant_df if service in PROMPT_INVARIANT_SERVICES else df_raw
    for _, row in src.iterrows():
        lc = row["language_code"]
        if lc == "en":
            continue
        val = row.get(col)
        if not isinstance(val, str) or not val.strip():
            continue
        if has_source_leakage(val, term):
            source_term_by_lang[lc].add(service)

# ── 5. Repetition loops, extreme term length, unicode escapes ────────────────
repeat_loop_by_lang = defaultdict(set)
extreme_len_by_lang = defaultdict(set)
unicode_esc_by_lang = defaultdict(set)
short_by_lang       = defaultdict(set)

for service, col in ALL_SVC_COLS.items():
    if col not in df_raw.columns:
        continue
    src = prompt_invariant_df if service in PROMPT_INVARIANT_SERVICES else df_raw
    for _, row in src.iterrows():
        val = row.get(col)
        if not isinstance(val, str) or not val.strip():
            continue
        lc = row["language_code"]
        if is_repetition_loop(val):
            repeat_loop_by_lang[lc].add(service)
        if has_extreme_term_length(val):
            extreme_len_by_lang[lc].add(service)
        if has_unicode_escape(val):
            unicode_esc_by_lang[lc].add(service)
        if has_short_translation(val):
            short_by_lang[lc].add(service)

# ── 5b. Language-name pass-through (term contains target language name) ──────
# Catches cases where the model returns the language name itself as the term,
# e.g. 'Mbere mbere mbere' for Mbere or 'Tasawaq baaɗaɗe' for Tasawaq. Applies
# to all variants and all services (LLM and direct).
lang_name_lookup = (
    df_raw[['language_code', 'language_name']]
    .drop_duplicates('language_code')
    .dropna(subset=['language_code'])
    .set_index('language_code')['language_name']
    .to_dict()
)
language_name_term_by_lang = defaultdict(set)
for service, col in ALL_SVC_COLS.items():
    if col not in df_raw.columns:
        continue
    src_df = prompt_invariant_df if service in PROMPT_INVARIANT_SERVICES else df_raw
    for _, row in src_df.iterrows():
        val = row.get(col)
        if not isinstance(val, str) or not val.strip():
            continue
        lc = row['language_code']
        lname = lang_name_lookup.get(lc, '')
        if lname and is_language_name_term(val, lname):
            language_name_term_by_lang[lc].add(service)

# ── 6. Any script mixing (including below exclusion threshold) ──────────────
# any_mixing fires whenever secondary-script chars are present at all.
# This is a superset of has_mixed_script (nulled) + has_romanization (stripped):
# it also captures sub-threshold cases that curate_translation leaves unchanged.
any_mixing_by_lang = defaultdict(set)

for service, col in ALL_SVC_COLS.items():
    if col not in df_raw.columns:
        continue
    src = prompt_invariant_df if service in PROMPT_INVARIANT_SERVICES else df_raw
    for _, row in src.iterrows():
        val = row.get(col)
        if not isinstance(val, str) or not val.strip():
            continue
        detail = script_mix_detail(val)
        if detail.get("any_mixing"):
            any_mixing_by_lang[row["language_code"]].add(service)

# ── Build DataFrame ───────────────────────────────────────────────────────────
lang_meta = (
    df_raw[["language_code", "language_name"]]
    .drop_duplicates("language_code")
    .dropna(subset=["language_code"])
)
lang_meta = lang_meta.copy()
lang_meta["language_family"] = lang_meta["language_code"].apply(get_language_family)

# Fix any language_name values that fell back to the language code
ref_path = os.path.join(DATA_DIR, "metadata_files", "language_codes_comprehensive.csv")
if os.path.exists(ref_path):
    _ref = pd.read_csv(ref_path, dtype=str).set_index("language_code")["language_name"]
    broken = lang_meta["language_name"] == lang_meta["language_code"]
    if broken.any():
        lang_meta.loc[broken, "language_name"] = (
            lang_meta.loc[broken, "language_code"].map(_ref)
            .fillna(lang_meta.loc[broken, "language_code"])
        )

flag_rows = []
for _, meta in lang_meta.iterrows():
    lc = meta["language_code"]
    m_svcs = sorted(missing_rat_by_lang.get(lc, set()))
    x_svcs = sorted(mixed_by_lang.get(lc, set()))
    r_svcs = sorted(roman_by_lang.get(lc, set()))
    d_svcs = sorted(disagr_by_lang.get(lc, set()))
    l_svcs = sorted(source_term_by_lang.get(lc, set()))
    p_svcs = sorted(placeholder_by_lang.get(lc, set()))
    rl_svcs = sorted(repeat_loop_by_lang.get(lc, set()))
    el_svcs = sorted(extreme_len_by_lang.get(lc, set()))
    ue_svcs = sorted(unicode_esc_by_lang.get(lc, set()))
    sh_svcs = sorted(short_by_lang.get(lc, set()))
    am_svcs = sorted(any_mixing_by_lang.get(lc, set()))
    rf_svcs = sorted(refusal_rat_by_lang.get(lc, set()))
    tr_svcs = sorted(transliteration_rat_by_lang.get(lc, set()))
    pr_svcs = sorted(placeholder_rat_by_lang.get(lc, set()))
    ln_svcs = sorted(language_name_term_by_lang.get(lc, set()))
    ur_svcs = sorted(unexp_rat_lang_by_lang.get(lc, set()))
    active = (
        (["missing_rationale"]   if m_svcs  else []) +
        (["mixed_script"]        if x_svcs  else []) +
        (["romanization"]        if r_svcs  else []) +
        (["script_disagreement"] if d_svcs  else []) +
        (["source_term"]         if l_svcs  else []) +
        (["placeholder_term"]    if p_svcs  else []) +
        (["repetition_loop"]     if rl_svcs else []) +
        (["extreme_term_length"] if el_svcs else []) +
        (["unicode_escape"]      if ue_svcs else []) +
        (["short_translation"]   if sh_svcs else []) +
        (["any_mixing"]          if am_svcs else []) +
        (["refusal_rationale"]   if rf_svcs else []) +
        (["transliteration_rationale"] if tr_svcs else []) +
        (["placeholder_rationale"] if pr_svcs else []) +
        (["language_name_term"] if ln_svcs else []) +
        (["unexpected_rationale_language"] if ur_svcs else [])
    )
    flag_rows.append({
        "language_code":               lc,
        "language_name":               meta["language_name"],
        "language_family":             meta["language_family"],
        "has_missing_rationale":       bool(m_svcs),
        "missing_rationale_services":  ";".join(m_svcs),
        "has_mixed_script":            bool(x_svcs),
        "mixed_script_services":       ";".join(x_svcs),
        "has_romanization":            bool(r_svcs),
        "romanization_services":       ";".join(r_svcs),
        "has_script_disagreement":     bool(d_svcs),
        "script_disagr_services":      ";".join(d_svcs),
        "has_source_term":             bool(l_svcs),
        "source_term_services":        ";".join(l_svcs),
        "has_placeholder_term":        bool(p_svcs),
        "placeholder_term_services":   ";".join(p_svcs),
        "has_repetition_loop":         bool(rl_svcs),
        "repetition_loop_services":    ";".join(rl_svcs),
        "has_extreme_term_length":     bool(el_svcs),
        "extreme_term_length_services":";".join(el_svcs),
        "has_unicode_escape":          bool(ue_svcs),
        "unicode_escape_services":     ";".join(ue_svcs),
        "has_short_translation":        bool(sh_svcs),
        "short_translation_services":  ";".join(sh_svcs),
        "has_any_mixing":               bool(am_svcs),
        "any_mixing_services":          ";".join(am_svcs),
        "has_refusal_rationale":              bool(rf_svcs),
        "refusal_rationale_services":         ";".join(rf_svcs),
        "has_transliteration_rationale":      bool(tr_svcs),
        "transliteration_rationale_services": ";".join(tr_svcs),
        "has_placeholder_rationale":          bool(pr_svcs),
        "placeholder_rationale_services":     ";".join(pr_svcs),
        "has_language_name_term":             bool(ln_svcs),
        "language_name_term_services":        ";".join(ln_svcs),
        "has_unexpected_rationale_language":  bool(ur_svcs),
        "unexpected_rationale_language_services": ";".join(ur_svcs),
        "automated_review_signals":   ";".join(active),
        # Backward-compatible alias for older notebooks/scripts.
        "quality_flags":               ";".join(active),
        "flag_count":                  len(active),
    })

quality_flags_df = (
    pd.DataFrame(flag_rows)
    .sort_values(["flag_count", "language_name"], ascending=[False, True])
    .reset_index(drop=True)
)

# ── Save ──────────────────────────────────────────────────────────────────────
flags_dir = os.path.join(DATA_DIR, "translated_terms", term.lower().replace(" ", "_"), "evaluation")
os.makedirs(flags_dir, exist_ok=True)
signals_path = os.path.join(flags_dir, "automated_review_signals.csv")
quality_flags_df.to_csv(signals_path, index=False)

flagged = quality_flags_df[quality_flags_df["flag_count"] > 0]
print(f"Saved: {signals_path}")
print(f"  Total languages  : {len(quality_flags_df)}")
print(f"  Flagged          : {len(flagged)} ({len(flagged)/len(quality_flags_df)*100:.1f}%)")
print()
for col, label in [
    ("has_missing_rationale", "Missing rationale "),
    ("has_mixed_script", "Mixed script above threshold"),
    ("has_romanization", "Romanization"),
    ("has_script_disagreement", "Script disagreement"),
    ("has_source_term", "Source term"),
    ("has_placeholder_term", "Placeholder term"),
    ("has_repetition_loop", "Repetition loop"),
    ("has_extreme_term_length", "Extreme length "),
    ("has_unicode_escape", "Unicode escape"),
    ("has_short_translation", "Short translation (<4 codepoints)"),
    ("has_any_mixing",   "Any mixing of scripts (including sub-threshold)"),
    ("has_refusal_rationale", "Refusal rationale (model admits no translation in rationale text)"),
    ("has_transliteration_rationale", "Transliteration rationale (model describes phonetic mapping)"),
    ("has_placeholder_rationale", "Placeholder rationale (model admits using a placeholder/stand-in)"),
    ("has_language_name_term", "Language-name pass-through (term contains target language name)"),
    ("has_unexpected_rationale_language", "Unexpected rationale language (English variant produced non-Latin rationale)"),
]:
    n   = int(quality_flags_df[col].sum())
    pct = n / len(quality_flags_df) * 100
    print(f"  {label}: {n:>3d}  ({pct:.1f}%)")


SIGNAL_GROUPS = {
    "pipeline_integrity": [
        "has_missing_rationale",
        "has_unexpected_rationale_language",
    ],
    "output_failure": [
        "has_mixed_script",
        "has_placeholder_term",
        "has_repetition_loop",
        "has_extreme_term_length",
        "has_unicode_escape",
        "has_refusal_rationale",
        "has_placeholder_rationale",
    ],
    "curation_search_complication": [
        "has_romanization",
        "has_source_term",
        "has_short_translation",
        "has_language_name_term",
        "has_transliteration_rationale",
    ],
    "analytical_signal": [
        "has_script_disagreement",
        "has_any_mixing",
    ],
}

signal_group_rows = []
for group, cols in SIGNAL_GROUPS.items():
    present = [c for c in cols if c in quality_flags_df.columns]
    if not present:
        continue
    n = int(quality_flags_df[present].any(axis=1).sum())
    signal_group_rows.append({
        "signal_group": group,
        "n_languages": n,
        "pct_languages": round(n / len(quality_flags_df) * 100, 1),
        "signals": "; ".join(c.replace("has_", "") for c in present),
    })

print("\nSignal groups:")
display(pd.DataFrame(signal_group_rows))

flag_count_dist = (
    quality_flags_df["flag_count"]
    .value_counts()
    .rename_axis("n_signals")
    .reset_index(name="n_languages")
    .sort_values("n_signals")
)
print("\nLanguages by number of automated review signals:")
display(flag_count_dist)

print("\nTop 10 flagged languages:")
print(quality_flags_df.head(10)[["language_name", "automated_review_signals", "flag_count"]].to_string(index=False))


Saved: /Users/zleblanc/CodingDH/translation_transmogrification_pipeline/datasets/translated_terms/digital_humanities/evaluation/automated_review_signals.csv
  Total languages  : 881
  Flagged          : 848 (96.3%)

  Missing rationale : 407  (46.2%)
  Mixed script above threshold:  94  (10.7%)
  Romanization:  89  (10.1%)
  Script disagreement: 376  (42.7%)
  Source term: 420  (47.7%)
  Placeholder term:  28  (3.2%)
  Repetition loop:  14  (1.6%)
  Extreme length :  21  (2.4%)
  Unicode escape:  10  (1.1%)
  Short translation (<4 codepoints):  24  (2.7%)
  Any mixing of scripts (including sub-threshold): 236  (26.8%)
  Refusal rationale (model admits no translation in rationale text): 687  (78.0%)
  Transliteration rationale (model describes phonetic mapping): 735  (83.4%)
  Placeholder rationale (model admits using a placeholder/stand-in):  50  (5.7%)
  Language-name pass-through (term contains target language name):  78  (8.9%)
  Unexpected rationale language (English variant produc

,signal_group,n_languages,pct_languages,signals
0,pipeline_integrity,556,63.1,missing_rationale; unexpected_rationale_language
1,output_failure,719,81.6,mixed_script; placeholder_term; repetition_loop; extreme_term_leng...
2,curation_search_complication,781,88.6,romanization; source_term; short_translation; language_name_term; ...
3,analytical_signal,411,46.7,script_disagreement; any_mixing



Languages by number of automated review signals:


,n_signals,n_languages
7,0,33
5,1,73
4,2,103
1,3,160
0,4,163
2,5,136
3,6,116
6,7,66
8,8,21
10,9,4



Top 10 flagged languages:
         language_name                                                                                                                                                                      automated_review_signals  flag_count
American Sign Language  script_disagreement;source_term;extreme_term_length;unicode_escape;short_translation;any_mixing;refusal_rationale;transliteration_rationale;language_name_term;unexpected_rationale_language          10
           Blissymbols                 missing_rationale;romanization;script_disagreement;source_term;repetition_loop;short_translation;any_mixing;refusal_rationale;transliteration_rationale;placeholder_rationale          10
             Khotanese             missing_rationale;mixed_script;romanization;script_disagreement;source_term;placeholder_term;any_mixing;refusal_rationale;transliteration_rationale;unexpected_rationale_language          10
    Large Flowery Miao missing_rationale;romanization;script_disagreement

### Error Categories × Automated Review Signals

The automated review signal file captures per-language output anomalies and review prompts (e.g. script mixing, source-term leakage, repetition loops); the error logs capture per-request service failures. Cross-tabulating these two dimensions reveals which pipeline failures tend to co-occur and highlights the most diagnostic language profiles.

Two patterns of particular interest are calculated live below:

**Honest-no doubles** — a language where one service returns a placeholder or undeciphered-script signal while another service honestly refuses (`extinct_ancient` or `knowledge_gap`). These are the most defensible exclusion decisions because multiple independent signals agree the translation is unreliable.

**Fabricate-vs-passthrough** — a language where one service produces a repetition-loop hallucination while another simply passes through the English source term. Both are failures of different kinds; neither is a usable translation.


In [19]:
# Error categories × automated review signals cross-tab
qf = pd.read_csv(
    os.path.join(DATA_DIR, "translated_terms", TARGET_TERMS[0].lower().replace(" ", "_"),
                 "evaluation", "automated_review_signals.csv"),
    converters={"language_code": str},
)
flag_cols = [c for c in qf.columns if c.startswith("has_")]
_err_cats  = ["max_tokens", "api_error", "empty_translation", "ollama_timeout",
              "extinct_ancient", "knowledge_gap", "generic_refusal",
              "repetition_loop", "parse_format", "other"]

# Per-language error-category set
lang_errcats = (
    llm_err.groupby("language_code")["category"]
    .apply(set)
    .reset_index()
    .rename(columns={"category": "err_cats"})
)
_cross = qf.merge(lang_errcats, on="language_code", how="left")
_cross["err_cats"] = _cross["err_cats"].apply(lambda x: x if isinstance(x, set) else set())

_rows = []
for flag in flag_cols:
    flagged = _cross[_cross[flag] == True]
    for ecat in _err_cats:
        n = flagged["err_cats"].apply(lambda s: ecat in s).sum()
        _rows.append({"flag": flag.replace("has_",""), "error_category": ecat, "n": int(n)})

xtab = pd.DataFrame(_rows)
xtab_wide = xtab.pivot(index="flag", columns="error_category", values="n").fillna(0).astype(int)
print("Automated review signals × error categories (count = languages with both signals)")
print(xtab_wide[_err_cats].to_string())

# Visualise as a heatmap
xtab_long = xtab.copy()
xtab_hm = alt.Chart(xtab_long[xtab_long["n"] > 0]).mark_rect().encode(
    x=alt.X("error_category:N", sort=_err_cats, title=None,
            axis=alt.Axis(labelAngle=-45)),
    y=alt.Y("flag:N", title=None),
    color=alt.Color("n:Q", scale=alt.Scale(scheme="oranges"), title="languages"),
    tooltip=["flag:N", "error_category:N", "n:Q"],
).properties(width=480, height=260, title="Automated review signals × error categories")
display(xtab_hm)

# ── Honest-no doubles ────────────────────────────────────────────────────────
honest_rows = []
ph_svc = qf[qf["has_placeholder_term"] == True][["language_code", "language_name", "placeholder_term_services"]]
for _, row in ph_svc.iterrows():
    lang = row["language_code"]
    ph_svcs = {svc.strip() for svc in str(row["placeholder_term_services"]).split(";") if svc.strip()}
    honest = llm_err[(llm_err["language_code"] == lang) &
                     (llm_err["category"].isin(["extinct_ancient", "knowledge_gap"])) &
                     (~llm_err["service"].isin(ph_svcs))]
    if not honest.empty:
        honest_rows.append({
            "language_code": lang,
            "language_name": row["language_name"],
            "placeholder_services": "; ".join(sorted(ph_svcs)),
            "honest_no_services": "; ".join(sorted(honest["service"].unique())),
            "honest_no_categories": "; ".join(sorted(honest["category"].unique())),
        })

honest_no_df = pd.DataFrame(honest_rows)
print(f"\nHonest-no doubles: {len(honest_no_df)} languages")
if not honest_no_df.empty:
    display(honest_no_df)

# ── Fabricate-vs-passthrough ─────────────────────────────────────────────────
fabricate_passthrough_df = qf[qf["has_repetition_loop"] & qf["has_source_term"]][
    ["language_code", "language_name", "repetition_loop_services", "source_term_services"]
].copy()
print(f"\nFabricate-vs-passthrough: {len(fabricate_passthrough_df)} languages")
if not fabricate_passthrough_df.empty:
    display(fabricate_passthrough_df)


Automated review signals × error categories (count = languages with both signals)
error_category                 max_tokens  api_error  empty_translation  ollama_timeout  extinct_ancient  knowledge_gap  generic_refusal  repetition_loop  parse_format  other
flag                                                                                                                                                                          
any_mixing                              6          5                  7               0                8             32               34               17             3     24
extreme_term_length                     2          1                  1               0                4              7                5                1             0      4
language_name_term                      4          1                  1               0                2             21               13                5             0     15
missing_rationale                      32  

alt.Chart(...)


Honest-no doubles: 15 languages


,language_code,language_name,placeholder_services,honest_no_services,honest_no_categories
0,egy,Ancient Egyptian,Gemini,Llama,extinct_ancient
1,cad,Caddo,OpenAI,Llama,knowledge_gap
2,elx,Elamite,Gemini,Llama; Mistral; OpenAI,extinct_ancient
3,ecy,Eteocypriot,Claude; Gemini,Llama; OpenAI,extinct_ancient; knowledge_gap
4,was,Washo,OpenAI,Llama,knowledge_gap
5,aro,Araona,DeepSeek; Gemini,OpenAI,knowledge_gap
6,kro,Kru languages,Claude,OpenAI,knowledge_gap
7,kut,Kutenai,Claude,Llama; OpenAI,knowledge_gap
8,clc,Chilcotin,OpenAI,Llama,knowledge_gap
9,hit,Hittite,Gemini; OpenAI,Llama,extinct_ancient; knowledge_gap



Fabricate-vs-passthrough: 12 languages


,language_code,language_name,repetition_loop_services,source_term_services
1,zbl,Blissymbols,Mistral,Claude
9,yav,Yangben,DeepSeek,Claude;DeepSeek;OpenAI
24,xsa,Sabaean,Gemma,Claude;DeepSeek
65,xlc,Lycian,Gemma,Claude;DeepSeek
71,xmr,Meroitic,Gemma,Claude;DeepSeek;Mistral
86,ter,Tereno,Gemma,Claude;DeepSeek;Gemini;OpenAI
137,hnj,Hmong Njua,Gemma,DeepSeek
153,kro,Kru languages,Llama,Claude;DeepSeek;OpenAI
179,ttm,Northern Tutchone,DeepSeek,Claude;OpenAI
194,tce,Southern Tutchone,DeepSeek,Claude;OpenAI


## 2.3 Manual Review & Exclusion Summary

The automated review signals in §2.2 surface *potential* problems; human review in the HTML explorer produces the authoritative exclusion decisions saved in `manual_exclusions.csv`.

Three exclusion types shape all downstream analysis:

- **`analysis_exclusion`** — translation is structurally unusable (loops, untranslatable scripts, pure error tokens). These languages are **dropped entirely** from notebooks 03–08.
- **`search_exclusion`** — translation is analytically interesting but unsafe as a GitHub search string (over-short terms, Braille, extreme repetition). Dropped only in notebook 09 (search results).
- **`term_correction`** — minor fixable error (spurious punctuation, source-term wrapper). Corrected form is used in string comparisons.

This section quantifies each type, cross-tabs them against the automated flags to measure auto-pipeline miss rate, and characterises patterns in manual term edits.

In [20]:
from scripts.utils import load_manual_exclusions

term = TARGET_TERMS[0]
eval_dir = os.path.join(DATA_DIR, "translated_terms", term.lower().replace(" ", "_"), "evaluation")
analysis_langs, search_terms, corrections = load_manual_exclusions(eval_dir)

excl_df = pd.read_csv(os.path.join(eval_dir, "manual_exclusions.csv"), dtype=str).fillna("")

print("=== Manual Exclusion Counts ===")
print(f"  analysis_exclusion : {len(analysis_langs):>4} unique language codes")
print(f"  search_exclusion   : {len(search_terms):>4} (language, term) pairs  "
      f"({excl_df['service'].eq('search_exclusion').sum()} rows, "
      f"{excl_df[excl_df['service']=='search_exclusion']['language_code'].nunique()} unique langs)")
print(f"  term_correction    : {len(corrections):>4} corrections")
print()

# Overlap: languages that appear in both analysis AND search exclusions
ae_langs = set(excl_df[excl_df["service"]=="analysis_exclusion"]["language_code"])
se_langs = set(excl_df[excl_df["service"]=="search_exclusion"]["language_code"])
print(f"  Overlap (both analysis + search exclusion): {len(ae_langs & se_langs)} languages")
print(f"  Total excluded from ≥1 analysis role      : {len(ae_langs | se_langs)} languages")
print(f"  Unaffected languages                       : {len(qf['language_code'].unique()) - len(ae_langs | se_langs)}")

=== Manual Exclusion Counts ===
  analysis_exclusion :   95 unique language codes
  search_exclusion   :  383 (language, term) pairs  (383 rows, 226 unique langs)
  term_correction    :   94 corrections

  Overlap (both analysis + search exclusion): 37 languages
  Total excluded from ≥1 analysis role      : 285 languages
  Unaffected languages                       : 596


### Auto-Pipeline Miss Rate

For each language that received a manual `analysis_exclusion`, we check whether the automated-review-signal pipeline would have caught it using the same likely-error tier applied by `filter_for_analysis(mode="quality")` and the review explorer's `xall` hints:

- **Caught**: the language had at least one likely-error signal (`has_repetition_loop`, `has_mixed_script`, `has_placeholder_term`, `has_unicode_escape`, `has_extreme_term_length`).
- **Missed**: the language had *zero* likely-error signals — the automated-review pipeline saw no unambiguous failure.

A high miss rate implies that manual review remains essential even after automated filtering.


In [21]:
qf = pd.read_csv(os.path.join(eval_dir, "automated_review_signals.csv"), converters={"language_code": str})

AUTO_CATCH_FLAGS = [
    "has_repetition_loop", "has_mixed_script", "has_placeholder_term",
    "has_unicode_escape", "has_extreme_term_length",
]
flag_cols_present = [c for c in AUTO_CATCH_FLAGS if c in qf.columns]

ae_qf = qf[qf["language_code"].isin(analysis_langs)].copy()
ae_qf["any_auto_flag"] = ae_qf[flag_cols_present].apply(
    lambda row: any(str(v).strip().lower() == "true" for v in row), axis=1
)
_caught = ae_qf["any_auto_flag"].sum()
missed = len(ae_qf) - _caught
miss_rate = missed / len(ae_qf) if len(ae_qf) else 0

print(f"Analysis-excluded languages: {len(ae_qf)}")
print(f"  Caught by likely-error signals : {_caught}  ({_caught/len(ae_qf):.0%})")
print(f"  Missed by likely-error signals : {missed}  ({miss_rate:.0%})")
print()
print("Missed languages (zero likely-error signals):")
missed_df = ae_qf[~ae_qf["any_auto_flag"]][["language_code", "language_name"] + flag_cols_present].reset_index(drop=True)
print(missed_df.to_string(index=False))
print()

# Which auto-flag types most often co-occur with analysis exclusions?
print("Likely-error signal co-occurrence with analysis_exclusion (among caught):")
for col in flag_cols_present:
    n = ae_qf[col].apply(lambda v: str(v).strip().lower() == "true").sum()
    print(f"  {col:<35} {n:>3}  ({n/len(ae_qf):.0%})")

Analysis-excluded languages: 95
  Caught by likely-error signals : 49  (52%)
  Missed by likely-error signals : 46  (48%)

Missed languages (zero likely-error signals):
language_code         language_name  has_repetition_loop  has_mixed_script  has_placeholder_term  has_unicode_escape  has_extreme_term_length
          shn                  Shan                False             False                 False               False                    False
          xna Ancient North Arabian                False             False                 False               False                    False
          bbj               Ghomala                False             False                 False               False                    False
          lab              Linear A                False             False                 False               False                    False
          nsk               Naskapi                False             False                 False               False     

### Term Correction Patterns

Each `term_correction` row records a manual edit to a translation. Categorising these edits reveals what kinds of artefacts the LLMs most consistently introduce.

In [22]:
import re

corr_df = excl_df[excl_df["service"] == "term_correction"].copy()
corr_df = corr_df[corr_df["corrected_term"].str.strip() != ""].reset_index(drop=True)

def classify_edit(orig, corr):
    orig, corr = str(orig).strip(), str(corr).strip()
    src_term = term  # "Digital Humanities"
    # Leading / trailing punctuation stripped
    if corr == re.sub(r'^[\s\W]+|[\s\W]+$', '', orig):
        return "strip_punctuation"
    # Source term wrapper removed  (e.g. "Digital Humanities (X)" → "X")
    if src_term in orig and src_term not in corr:
        return "remove_source_wrapper"
    # Leading slash removed  (e.g. "/Dkotan" → "Dkotan")
    if orig.startswith("/") and corr == orig[1:]:
        return "remove_leading_slash"
    # Parenthetical stripped from end
    if re.search(r'\(.+\)$', orig) and corr == re.sub(r'\s*\(.+\)$', '', orig).strip():
        return "strip_trailing_parenthetical"
    # Transliteration / script cleanup (corrected has different script)
    return "other"

corr_df["edit_type"] = corr_df.apply(
    lambda r: classify_edit(r["original_term"], r["corrected_term"]), axis=1
)

print("Term correction edit types:")
print(corr_df["edit_type"].value_counts().to_string())
print(f"\nTotal corrections: {len(corr_df)}")
print()
print("Examples per edit type:")
for etype, grp in corr_df.groupby("edit_type"):
    ex = grp[["language_code", "original_term", "corrected_term"]].head(3)
    print(f"\n  [{etype}]")
    for _, r in ex.iterrows():
        print(f"    {r['language_code']}: {repr(r['original_term'][:60])} → {repr(r['corrected_term'][:60])}")

Term correction edit types:
edit_type
other                           53
strip_trailing_parenthetical    20
strip_punctuation               12
remove_source_wrapper            7
remove_leading_slash             2

Total corrections: 94

Examples per edit type:

  [other]
    ain: 'ペーシ・ウシ・イプキ・アプ・シケペ' → 'Dkara Usaamrra'
    brx: 'digitāl ḥumnāt' → 'ཐོགས་པའི་བདེན་ཆུང'
    brx: 'DIGITAL HUMANITIES' → 'ဗိုႯလ်ချမ်းဘာသာ'

  [remove_leading_slash]
    gld: '/Dkaw ᑕ+-+-+' → 'Dkaw ᑕ+-+-+'
    eky: '/Dkaw ၲေတာင္းမြန်းအစား' → 'Dkaw ၲေတာင္းမြန်းအစား'

  [remove_source_wrapper]
    brx: 'Digital Humanities ≈ ཐོགས་པའི་བདེན་ཆུང' → 'ཐོགས་པའི་བདེན་ཆུང'
    brx: 'Digital Humanities ဗိုႯလ်ချမ်းဘာသာ (Dịch vụ người lập trình ' → 'ဗိုႯလ်ချမ်းဘာသာ'
    crg: 'Digital Humanities naskapiy-owinimakan-asiniy-iskwēwak kiske' → 'naskapiy-owinimakan-asiniy-iskwēwak kiskeya-mīna-oyawiyik'

  [strip_punctuation]
    ain: '/Dkara Usaamrra' → 'Dkara Usaamrra'
    ain: '/Dkiritik Huamuuni' → 'Dkiritik Huamuuni'
    lus: '